<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/New-Baseline-v2/mnps_new_baseline%20v7.0.1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MNPS Job Equity New Baseline 7.0.1**
> A notebook to help you get started  
> DSI DSSG + MNPS   
> September 30, 2025  
> Drafted by Wayne Birch - [contact him](wayne.birch@mnps.org) for questions, code update needs, or other questions about the notebook!

This notebook is a restart point based on the work done post mini Hackathon with Metro Nashville Public Schools (MNPS) and the VU Data Science Institute (VU DSI).





## **2** | Environment Setup
We provide this code just as a rapid method to get started, and focus our efforts on implementation through Google Colab.

### **2a** | API Key Setup
#### **2a.1** | Access
The DSI has provided you an API key which can access **some** of the OpenAI models. These include:
* All versions of gpt-4o
* All versions of gpt-4.1
* All versions of o3-mini

Vector store upload, web search, code interpreter, and other functionality outside of the Chat Completions and Messages API is **not** supported. If you really want to use these things, you will have to make a good and cost-supported argument. If you don't feel like arguing, you can also utilize your own OpenAI API key.

#### **2a.2** | API Keys in Google Colab
To use your API key, click on the key icon (looks sort of like 🔑) in the left sidebar.  Under **Name**, add `OPENAI_API_KEY`. Under **Value**, paste your API key. Your API key is a jumble of numbers and letters, maybe even other symbols. Click the slider checkbox to enable **Notebook access** (so your notebook will grab these values without asking you).  

### **2b** | Runtime setup
We're going to install some packages in your environment so that you have access to the code functionality. If you need more packages, install more packages. Install **only** packages you trust.


> # **Version 6.0 Change**
> Added second API call to determine confidence level of each record against each roll.


In [1]:
#Cell 3
!pip install -U openai

In [2]:
#Cell 3.5
# ===== Environment Setup (single source of truth) =====
import os
from typing import List
import pandas as pd
from pydantic import BaseModel, Field
from google.colab import userdata

# 1) API key from Colab's 🔑 panel
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# 2) Read the model selector from Colab's 🔑 panel (can be alias or snapshot)
RAW_MODEL = userdata.get("OPENAI_MODEL")  # e.g., gpt-4o, gpt-4o-2024-11-20, gpt4.1, o3 mini

def normalize_model_id(s: str | None) -> str | None:
    if not s:
        return None
    s = s.strip().lower().replace("_", "-").replace(" ", "-")
    fixes = {
        "gpt4o": "gpt-4o",
        "gpt-4o": "gpt-4o",
        "gpt4.1": "gpt-4.1",
        "gpt-41": "gpt-4.1",
        "o3mini": "o3-mini",
        "o3-mini": "o3-mini",
    }
    return fixes.get(s, s)

alias_or_snapshot = normalize_model_id(RAW_MODEL)

# 3) Map aliases → pinned snapshots you prefer (edit to taste)
SNAPSHOTS = {
    # GPT-4o snapshots (stable; good for Structured Outputs)
    "gpt-4o":  "gpt-4o-2024-11-20",
    # GPT-4.1 family snapshot (long context)
    "gpt-4.1": "gpt-4.1-2025-04-14",
    # Keep o3-mini as an alias (no public dated snapshot ID); good for reasoning
    "o3-mini": "o3-mini",
}

# 4) Final MODEL_ID selection rule:
#    - If user entered an alias, pin it via SNAPSHOTS
#    - If user entered a snapshot, pass it through
#    - Else fallback to a safe default snapshot
MODEL_ID = SNAPSHOTS.get(alias_or_snapshot or "", None) or (alias_or_snapshot) or "gpt-4o-2024-11-20"

print("🔧 OPENAI_MODEL (raw):", RAW_MODEL)
print("✅ Using MODEL_ID:", MODEL_ID)


🔧 OPENAI_MODEL (raw): GPT-4o
✅ Using MODEL_ID: gpt-4o-2024-11-20


In [3]:
# ===== Cell 3.9 — Version banner & quick sanity =====
from pathlib import Path
import glob, sys

# <-- set this each time you save a new notebook -->
NOTEBOOK_VERSION = "v7.0.1"

req_symbols = ["full_text_for_row", "OUTPUTS_DIR"]
print(f"Notebook version: {NOTEBOOK_VERSION}")
for s in req_symbols:
    print(f"{s} defined:", s in globals())

print("BATCH_INPUT_CSV:", globals().get("BATCH_INPUT_CSV"))
print("MODEL_ID:", globals().get("MODEL_ID"))

# Show newest outputs so you can confirm you’re writing/reading the right run folder
odir = Path(globals().get("OUTPUTS_DIR","./outputs"))
odir.mkdir(parents=True, exist_ok=True)
hits = []
for pat in ["Job_Classifications_Batch*.csv", "classified_job_descriptions*.csv",
            "role_confidence_top5.*", "prompt_audit.csv", "postrun_sanity.md", "run_quality_report.md"]:
    hits += glob.glob(str(odir / pat))
for p in sorted(map(Path, hits), key=lambda p: p.stat().st_mtime, reverse=True)[:10]:
    print(f"{p.name:40s}  {p.stat().st_size:>8d} bytes")


Notebook version: v7.0.1
full_text_for_row defined: False
OUTPUTS_DIR defined: False
BATCH_INPUT_CSV: None
MODEL_ID: gpt-4o-2024-11-20


In [4]:
# ===== Cell 4 — Unique run folder + get inputs (3 files) + robust CSV read + upload to OpenAI =====
import os, json, shutil, datetime as dt, zipfile
from pathlib import Path
import pandas as pd
from google.colab import drive
from openai import OpenAI

# ---------- 0) Mount Drive ----------
drive.mount('/content/drive')

# ---------- 1) Fixed output location (as requested) ----------
RUN_ROOT = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
timestamp = dt.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
RUN_DIR = RUN_ROOT / f"RUN_{timestamp}"
INPUTS_DIR = RUN_DIR / "inputs"
OUTPUTS_DIR = RUN_DIR / "outputs"
for p in (RUN_DIR, INPUTS_DIR, OUTPUTS_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("🗂️ Run folder:", RUN_DIR)

# ---------- 2) Where to find your three inputs by default ----------
# If you want to upload instead of copying from Drive, set ALLOW_UPLOAD = True.
DATA_INPUTS_DIR = Path("/content/drive/My Drive/Colab Notebooks/Data Inputs")
ALLOW_UPLOAD = False  # set True to be prompted to upload the 3 files from your computer

REQUIRED = {
    "Ground Truth Masterfile.csv": DATA_INPUTS_DIR / "Ground Truth Masterfile.csv",
    "Sample JDs.csv":  DATA_INPUTS_DIR / "Sample JDs.csv",
    "MNPS_Prompt_Resources.zip":  DATA_INPUTS_DIR / "MNPS_Prompt_Resources.zip",
}

# (A) Optionally upload files instead of copying from Drive
if ALLOW_UPLOAD:
    from google.colab import files as colab_files
    print("🔼 Upload the three files when prompted:")
    uploaded = colab_files.upload()  # opens a browser picker
    for name in REQUIRED.keys():
        if name in uploaded:
            dst = INPUTS_DIR / name
            with open(dst, "wb") as f:
                f.write(uploaded[name])
            REQUIRED[name] = dst  # point to the just-uploaded copy

# (B) Copy from Drive if not already present in /inputs
missing = []
for name, src in REQUIRED.items():
    dst = INPUTS_DIR / name
    if dst.exists():
        continue
    if src.exists():
        shutil.copy2(src, dst)
        print(f"📄 Copied: {src}  →  {dst}")
    else:
        missing.append(name)

if missing:
    raise FileNotFoundError(
        "These input files were not found. Place them in "
        f"{DATA_INPUTS_DIR} or enable ALLOW_UPLOAD=True:\n - " + "\n - ".join(missing)
    )

# ---------- 3) Unpack the resources zip into inputs/resources (optional but helpful) ----------
resources_zip = INPUTS_DIR / "MNPS_Prompt_Resources.zip"
RESOURCES_DIR = INPUTS_DIR / "resources"
if resources_zip.exists():
    RESOURCES_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(resources_zip, "r") as zf:
        zf.extractall(RESOURCES_DIR)
    print("🧰 Unpacked resources to:", RESOURCES_DIR)

# ---------- 4) Robust CSV reader (handles cp1252/latin1) ----------
def read_csv_smart(path: Path, **kwargs) -> pd.DataFrame:
    trials = [
        dict(encoding="utf-8"),
        dict(encoding="utf-8-sig"),
        dict(encoding="cp1252"),
        dict(encoding="latin1"),
    ]
    for t in trials:
        try:
            df = pd.read_csv(path, **{**t, **kwargs})
            print(f"✅ Read {path.name} with encoding={t['encoding']}")
            return df
        except UnicodeDecodeError:
            continue
    # last resort
    df = pd.read_csv(path, encoding="latin1", on_bad_lines="skip", **kwargs)
    print(f"⚠️ Read {path.name} with encoding=latin1 (on_bad_lines='skip')")
    return df

# Smoke test: load one row from the sample CSV (row 0) and build job_desc_text for downstream cells
sample_csv = INPUTS_DIR / "Sample JDs.csv"
df = read_csv_smart(sample_csv)

required_cols = [
    "Job Description Name","Position Summary","Education","Work Experience",
    "Essential Functions","Licenses and Certifications","Knowledge, Skills and Abilities"
]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in {sample_csv.name}: {missing_cols}")

ROW_IDX = 0
r = df.iloc[ROW_IDX]
job_desc_text = f"""Position Summary: {r['Position Summary']}
Education: {r['Education']}
Work Experience: {r['Work Experience']}
Licenses and Certifications: {r['Licenses and Certifications']}
Essential Functions: {r['Essential Functions']}
Knowledge, Skills and Abilities: {r['Knowledge, Skills and Abilities']}
"""
print("🧪 Prepared job_desc_text from row", ROW_IDX)

# ---------- 5) Upload the two CSVs to OpenAI so later cells can attach them ----------
client = OpenAI()  # API key already set in your Environment Setup cell
to_upload = [
    INPUTS_DIR / "Ground Truth Masterfile.csv",
    INPUTS_DIR / "Sample JDs.csv",
]
uploaded = []
for p in to_upload:
    with open(p, "rb") as f:
        up = client.files.create(file=f, purpose="assistants")
    uploaded.append(up)

file_ids = [u.id for u in uploaded]  # <-- used by the Responses API cell later
print("⬆️ Uploaded file_ids:", file_ids)

# ---------- 6) Write a small manifest so you can audit each run ----------
manifest = {
    "run_folder": str(RUN_DIR),
    "created_utc": timestamp,
    "inputs": [str(p) for p in (INPUTS_DIR / "Ground Truth Masterfile.csv",
                                 INPUTS_DIR / "Sample JDs.csv")],
    "resources_dir": str(RESOURCES_DIR) if RESOURCES_DIR.exists() else None,
    "uploaded_file_ids": file_ids,
}
(RUN_DIR / "RUN_METADATA.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("\n📁 Current run tree (first few entries):")
for i, p in enumerate(sorted(RUN_DIR.rglob("*"))):
    print(" -", p.relative_to(RUN_DIR))
    if i > 25:
        print(" … (truncated)")
        break

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🗂️ Run folder: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251001_165753
📄 Copied: /content/drive/My Drive/Colab Notebooks/Data Inputs/Ground Truth Masterfile.csv  →  /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251001_165753/inputs/Ground Truth Masterfile.csv
📄 Copied: /content/drive/My Drive/Colab Notebooks/Data Inputs/Sample JDs.csv  →  /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251001_165753/inputs/Sample JDs.csv
📄 Copied: /content/drive/My Drive/Colab Notebooks/Data Inputs/MNPS_Prompt_Resources.zip  →  /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251001_165753/inputs/MNPS_Prompt_Resources.zip
🧰 Unpacked resources to: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251001_165753/inputs/resources
✅ Read Sample JDs.csv with encoding=utf-8
🧪 Prepared job_desc_text from row 0


/tmp/ipython-input-1838845499.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = dt.datetime.utcnow().strftime("%Y%m%d_%H%M%S")


⬆️ Uploaded file_ids: ['file-PnbQJ2nyKG6D428MwgZW3D', 'file-8Hrfzs8xsotW7rqKPR7QMy']

📁 Current run tree (first few entries):
 - RUN_METADATA.json
 - inputs
 - inputs/Ground Truth Masterfile.csv
 - inputs/MNPS_Prompt_Resources.zip
 - inputs/Sample JDs.csv
 - inputs/resources
 - inputs/resources/Competency Extended Descriptions.csv
 - inputs/resources/Korn_Ferry Lominger 38 Competencies.csv
 - inputs/resources/MNPS KSACs.csv
 - inputs/resources/MNPS Roles.csv
 - outputs


In [5]:
# Cell 6
from openai import OpenAI
client = OpenAI()

visible = {m.id for m in client.models.list().data}
if MODEL_ID not in visible:
    print(f"⚠️ {MODEL_ID} is not visible to your key. "
          "Use an alias you do see (e.g., gpt-4o) or confirm access in your org.")
else:
    print(f"👍 {MODEL_ID} is available.")


👍 gpt-4o-2024-11-20 is available.


## **3** | The Data

The current prompt is a two-step prompt that is successful through the ChatGPT interface. It requires two types of data:
* The data to be classified
* Supporting resources

We need to read all of this in. Let's grab it and use it. The first thing you'll do is just straight up download a zip file of all of this information.

You can download all of the reference files from the link provided, then upload in the sidebar. You'll then unzip the directory using the code below.

Click on the folder icon in the left sidebar (kinda looks like this 🗂️) and you'll see all the files there. We'll read them in.

# **Version 6.0**
> Pulls data input fro mounted Google drive folder and unzips for use in /content/  

In [6]:
# Cell 8
from google.colab import drive
drive.mount('/content/drive')
base_target_folder = '/content/drive/My Drive/Colab Notebooks/Data Inputs'
!unzip "{base_target_folder}/MNPS_Prompt_Resources.zip" -d /content/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Archive:  /content/drive/My Drive/Colab Notebooks/Data Inputs/MNPS_Prompt_Resources.zip
  inflating: /content/Korn_Ferry Lominger 38 Competencies.csv  
  inflating: /content/Competency Extended Descriptions.csv  
  inflating: /content/MNPS KSACs.csv  
  inflating: /content/MNPS Roles.csv  


## **4** | The Prompts

What we have here is a direct prompt to get the response that we're looking for. We'll make this happen directly using the OpenAI Chat Completions API. Note that you can use other APIs as you like.

In [7]:
#Cell 12
zero_shot_prompt = \
""" Objective: Evaluate and group jobs from the "New Sample_08.07.2025.csv" file based on similarities in job functions, not job titles.

Process:

- Compare all jobs against each other using the attributes listed in the file: Education, Work Experience, Licenses/Certifications, Essential Functions, Knowledge, Skills, Abilities, and Position Summary.
- Compare each job with reference sources using the same attributes. I have attached the reference sources for you.
- Group jobs based on similarities into:
  - Major role groupings (e.g., Specialist, Analyst, Manager)
  - Minor sub-groupings (e.g., I, II, III, IV) - not to exceed level IV
- Use the MNPS Roles and MNPS KSACs documents to help you determine major role groupings.
- Use the remaining documents to help you clarify subtle differences in role groupings and sub-groupings.
- Use a more qualitative, holistic assessment focused on functional alignment with KSACs rather than a quantitative scoring approach with defined complexity metrics

Output Format:

- Create a table with the following columns:
  - Original Job Title
  - New Job Title
  - Major Role Group
  - Minor Sub-Group
  - Justification for Grouping

- Provide an accompanying narrative explaining the rationale behind the groupings and any notable patterns or insights discovered during the analysis.

Job Title Convention:

- Follow the format: "[Function] [Role] [Level]" (e.g., "Collections Specialist II", "Accounts Payable Specialist III")

Additional Guidelines:

- Ensure all sources used are cited properly.
- Focus on the nature of the work performed rather than just the job titles.
- Consider the complexity of tasks, level of responsibility, and required competencies when determining groupings.
- Provide clear explanations for why each job was classified as it was, referencing specific job attributes and external benchmarks.

"""

In [8]:
#Cell 14
from pydantic import BaseModel, Field

class JobClassification(BaseModel):
    """Represents the classification of a job based on its functions."""
    job_title_original: str = Field(..., description="The original job title as provided in the input data using the job title convention specified.")
    new_job_title: str = Field(..., description="The proposed new job title based on the classification using the job title convention specified.")
    major_role_group: str = Field(..., description="The major grouping of the job based on its functional role (e.g., Specialist, Analyst, Manager).")
    minor_sub_group: str = Field(..., description="The minor sub-grouping within the major role group (e.g., Specialist I, II, III, IV).")
    grouping_justification: str = Field(..., description="The justification for placing the job in the specific major and minor groups, referencing job attributes and relevant documents.")

In [9]:
# Cell 14.9 — Load MNPS Roles & KSACs (closed set)

from pathlib import Path
import pandas as pd, io, os, re
from collections import OrderedDict

# (Optional) set explicit paths if you know them; otherwise auto-detect:
ROLES_CSV  = globals().get("ROLES_CSV",  None)  # e.g., "/content/MNPS Roles.csv"
KSACS_CSV  = globals().get("KSACS_CSV",  None)  # e.g., "/content/MNPS KSACs.csv"

# --- version-safe reader (no `errors=` kw) ---
def _read_csv_robust(path: str) -> pd.DataFrame:
    encs = ["utf-8","utf-8-sig","cp1252","latin1","windows-1252"]
    for enc in encs:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            pass
    with open(path, "rb") as f:
        raw = f.read()
    for enc in encs + ["latin1"]:
        try:
            txt = raw.decode(enc, errors="ignore")
            return pd.read_csv(io.StringIO(txt))
        except Exception:
            pass
    return pd.read_csv(path, engine="python")

def _auto_find(*names):
    roots = [Path(globals().get("OUTPUTS_DIR",".")), Path.cwd(), Path("/content"), Path("/content/drive/My Drive")]
    hits = []
    for r in roots:
        try:
            for nm in names:
                for p in r.rglob(nm):
                    if p.is_file():
                        hits.append(p)
        except Exception:
            pass
    if not hits:
        return None
    # newest first, then shallower
    hits = sorted(hits, key=lambda p: (-p.stat().st_mtime, str(p).count(os.sep)))
    return str(hits[0])

def _pick_col(cols, *patterns):
    for c in cols:
        cl = c.lower()
        for pat in patterns:
            if re.search(pat, cl):
                return c
    return None

# --- Locate files ---
if not ROLES_CSV or not Path(str(ROLES_CSV)).exists():
    ROLES_CSV = _auto_find("MNPS Roles.csv") or _auto_find("MNPS_Roles.csv")
if not KSACS_CSV or not Path(str(KSACS_CSV)).exists():
    KSACS_CSV = _auto_find("MNPS KSACs.csv") or _auto_find("MNPS_KSACs.csv")

if not ROLES_CSV or not Path(ROLES_CSV).exists():
    raise FileNotFoundError("Could not locate 'MNPS Roles.csv'. Set ROLES_CSV to its path.")
if not KSACS_CSV or not Path(KSACS_CSV).exists():
    print("[Warn] 'MNPS KSACs.csv' not found — proceeding without KSAC blobs.")
    KSACS_CSV = None

# --- Load Roles ---
df_roles = _read_csv_robust(ROLES_CSV)
role_col = _pick_col(df_roles.columns, r"^role$", r"major.*role", r"major.*group", r"mnps.*role")
if not role_col:
    # fall back to first column
    role_col = df_roles.columns[0]

# Build VALID_ROLES (preserve first-seen order, drop blanks)
_valid_roles = []
seen = set()
for val in df_roles[role_col].astype(str).fillna("").tolist():
    name = val.strip()
    if not name:
        continue
    if name not in seen:
        seen.add(name)
        _valid_roles.append(name)

if not _valid_roles:
    raise ValueError("MNPS Roles file loaded but produced an empty role list.")

# --- Load KSACs (role -> concatenated KSAC text) ---
role_ksac = {}
if KSACS_CSV:
    df_ks = _read_csv_robust(KSACS_CSV)
    # try to identify columns
    ks_role_col = _pick_col(df_ks.columns, r"^role$", r"major.*role", r"role.*name")
    ks_text_cols = [c for c in df_ks.columns
                    if re.search(r"(ksac|knowledge|skills|abilities|competenc|description)", c, flags=re.I)]
    if not ks_role_col:
        # if we can't find a role column, bail gracefully
        ks_role_col = df_ks.columns[0]
    if not ks_text_cols:
        # fall back to all non-role columns
        ks_text_cols = [c for c in df_ks.columns if c != ks_role_col]

    # raw map by KSAC file's role labels
    tmp_map = {}
    for _, r in df_ks.iterrows():
        rk = str(r.get(ks_role_col, "") or "").strip()
        if not rk:
            continue
        parts = []
        for c in ks_text_cols:
            v = str(r.get(c, "") or "").strip()
            if v:
                parts.append(v)
        if not parts:
            continue
        blob = " ".join(parts)
        tmp_map.setdefault(rk, []).append(blob)

    # collapse lists
    tmp_map = {k: " ".join(vs) for k, vs in tmp_map.items()}

    # Align KSAC labels to the closed set names.
    # If KSAC uses variants like "Budget Partner" but "Partner" is the official role,
    # attach the KSAC text to the official role when it is a substring match.
    role_ksac = {r: "" for r in _valid_roles}
    for ks_label, blob in tmp_map.items():
        matched = False
        for official in _valid_roles:
            if official.lower() == ks_label.lower() or official.lower() in ks_label.lower() or ks_label.lower() in official.lower():
                role_ksac[official] = (role_ksac.get(official, "") + " " + blob).strip()
                matched = True
                break
        if not matched:
            # keep unmapped KSACs separate (rare); not harmful
            role_ksac[ks_label] = role_ksac.get(ks_label, "") + " " + blob

# --- Expose globals used by 15.2 and 16 ---
VALID_ROLES = _valid_roles  # closed set
globals()["VALID_ROLES"] = VALID_ROLES
globals()["role_ksac"] = role_ksac

print(f"[Closed Set] Loaded {len(VALID_ROLES)} roles from: {ROLES_CSV}")
print("  Example roles:", ", ".join(VALID_ROLES[:10]), ("..." if len(VALID_ROLES) > 10 else ""))
print(f"[Closed Set] KSAC blobs attached for {sum(bool(v) for v in role_ksac.values())} roles.")


[Closed Set] Loaded 64 roles from: /content/drive/MyDrive/Colab Notebooks/Run Results/RUN_20251001_165753/inputs/resources/MNPS Roles.csv
  Example roles: Accountant, Administrative Assistant, Advisor, Agent, Aide, Analyst, Architect (Technology-Focused), Associate, Assistant, Auditor ...
[Closed Set] KSAC blobs attached for 60 roles.


In [10]:
# ===== Cell 14.95 — Leadership & Coach lexicon + canonicalization augment =====

# Extend/ensure the closed role set includes these roles explicitly
BASE_VALID_ROLES = list(globals().get("VALID_ROLES", []))
MUST_HAVE = ["Principal", "Assistant Principal", "Coach"]  # non-athletic Instructional Coach
VALID_ROLES = sorted(set(BASE_VALID_ROLES) | set(MUST_HAVE))
globals()["VALID_ROLES"] = VALID_ROLES

# Synonyms / normalizations used across the notebook (JSON backfill, reports, etc.)
ROLE_SYNONYMS = dict(globals().get("ROLE_SYNONYMS", {}))
ROLE_SYNONYMS.update({
    # Assistant Principal
    "ap": "Assistant Principal",
    "asst principal": "Assistant Principal",
    "assistant-principal": "Assistant Principal",
    "associate principal": "Assistant Principal",
    # Principal
    "school principal": "Principal",
    # Coach (non-athletic / instructional)
    "instructional coach": "Coach",
    "academic coach": "Coach",
    "learning coach": "Coach",
    "teaching coach": "Coach",
    "teacher coach": "Coach",
    "coaching specialist": "Coach",
    # Athletic disambiguators (we map these *away* later)
    "athletic coach": "Athletic Coach",  # not in VALID_ROLES; used as a filter signal
    "sports coach": "Athletic Coach",
    "head coach": "Athletic Coach",
    "assistant coach": "Athletic Coach",
})
globals()["ROLE_SYNONYMS"] = ROLE_SYNONYMS

# Lightweight keyword lexicon to detect leadership / instructional coaching vs athletic
LEADERSHIP_TOKENS = set("""
school leadership building leader admin leadership administrator leadership team
teacher evaluation evals observations walkthroughs discipline parent relations
school improvement sip mtss budgeting staffing scheduling master schedule
compliance accreditation title ix 504 iep attendance truancy safety drills
""".split())

AP_TOKENS = set("""
assistant principal ap dean discipline attendance testing coordinator
""".split())

PRINCIPAL_TOKENS = set("""
principal building principal head of school
""".split())

COACH_TOKENS = set("""
instructional coaching plc professional development pd model lessons co-teach
coaching cycles observation feedback data meetings curriculum alignment
""".split())

ATHLETIC_TOKENS = set("""
athletic athletics sport sports team varsity junior varsity jv tournament
game practice weight room field court track locker referee umpire
""".split())

globals().update({
    "LEADERSHIP_TOKENS": LEADERSHIP_TOKENS,
    "AP_TOKENS": AP_TOKENS,
    "PRINCIPAL_TOKENS": PRINCIPAL_TOKENS,
    "COACH_TOKENS": COACH_TOKENS,
    "ATHLETIC_TOKENS": ATHLETIC_TOKENS,
})
print("[14.95] VALID_ROLES now includes:", ", ".join(sorted(MUST_HAVE)))
print("[14.95] Synonyms loaded for Principal / Assistant Principal / Coach (non-athletic).")


[14.95] VALID_ROLES now includes: Assistant Principal, Coach, Principal
[14.95] Synonyms loaded for Principal / Assistant Principal / Coach (non-athletic).


In [11]:
#Cell 15
from typing import List

class JobClassificationTable(BaseModel):
  """The table classification and overall commentary on the groupings provided by the AI system."""
  job_classification_table: List[JobClassification] = Field(..., description="The table of job classifications.")
  narrative_rationale: str = Field(..., description="The narrative commentary on the groupings provided by the AI system.")

In [12]:
# Cell 15.0 — Role Confidence Output Schema (v6.0)

from typing import List, Dict, Optional
try:
    from pydantic import BaseModel, Field
except Exception as e:
    raise ImportError("Pydantic is required. Install with: pip install pydantic") from e

class RoleConfidence(BaseModel):
    role: str = Field(..., description="One MNPS major role from the closed set.")
    confidence: float = Field(..., ge=0.0, le=1.0, description="Confidence in [0,1].")
    rationale: str = Field(..., description="Brief reason; cite determinant factors/KSACs.")

class RoleConfidenceTable(BaseModel):
    row_id: int = Field(..., description="Row id of the input record (0-based).")
    job_description_name: str = Field(..., description="Original Job Description Name for this row.")
    confidences: List[RoleConfidence] = Field(
        ..., description="Confidence for each MNPS role (ideally all roles)."
    )
    top_roles_summary: Optional[str] = Field(
        None, description="Optional 1-3 sentence summary of the top distinctions."
    )

# Pydantic v1 shim
if not hasattr(RoleConfidenceTable, "model_json_schema"):
    RoleConfidenceTable.model_json_schema = classmethod(lambda cls, *a, **k: cls.schema())


In [13]:
# Cell 15.1 — Role Confidence Prompt Builder (v6.0)

import json
import textwrap

# Expect these exist from earlier cells; we will degrade gracefully.
VALID_ROLES = globals().get("VALID_ROLES", [])
role_ksac = globals().get("role_ksac", {})  # dict: role -> KSAC blob string

ROLE_CONFIDENCE_MODEL = globals().get("ROLE_CONFIDENCE_MODEL", globals().get("MODEL", "gpt-4o-2024-11-20"))
ROLE_CONFIDENCE_TEMP  = float(globals().get("ROLE_CONFIDENCE_TEMP", 0.2))

def _ksac_blurb_for_role(role: str) -> str:
    blob = role_ksac.get(role, "")
    return f"- {role}: {blob[:500]}{'...' if len(blob) > 500 else ''}"

def build_role_confidence_prompt(row, valid_roles=None):
    """Builds the prompt for the first API call (confidence by role)."""
    vr = valid_roles or VALID_ROLES
    if not vr:
        raise ValueError("VALID_ROLES is empty/undefined. Load MNPS Roles earlier.")

    # Determinant fields (robust access)
    def g(col): return str(row.get(col, "") or "")
    jd_name = g("Job Description Name")
    pos_sum = g("Position Summary")
    ess_fn  = g("Essential Functions")
    ksa     = g("Knowledge, Skills and Abilities")
    edu     = g("Education")
    exp     = g("Work Experience")
    lic     = g("Licenses and Certifications")

    ksac_section = "\n".join(_ksac_blurb_for_role(r) for r in vr[:64])

    instructions = f"""
You are evaluating one MNPS job against the CLOSED SET of MNPS major role groupings.

CLOSED SET (64 roles max, from MNPS Roles):
{", ".join(vr)}

KSAC Guidance (from MNPS KSACs; truncated where long):
{ksac_section}

Determinant Factors for this job:
- Job Description Name: {jd_name}
- Position Summary: {pos_sum}
- Essential Functions: {ess_fn}
- Knowledge, Skills and Abilities: {ksa}
- Education: {edu}
- Work Experience: {exp}
- Licenses and Certifications: {lic}

TASK:
For EACH role in the CLOSED SET, estimate a confidence in [0,1] for how well the job aligns to that role, based on KSAC fit and the determinant factors.
- Enforce eligibility minima implicitly: a role that clearly fails required Education/Experience/License should have very low confidence.
- Distinguish Analyst vs Specialist (analytics/metrics vs execution/process), Coordinator vs Technician, and Supervisor/Manager/Director by scope/people management/strategy.
- Keep rationales concise.

OUTPUT:
Return ONLY valid JSON that matches this JSON Schema:

{json.dumps(RoleConfidenceTable.model_json_schema(), indent=2)}
""".strip()

    return instructions


In [14]:
# ===== Cell 15.19 — Reset confidence state =====
from pathlib import Path

OUTPUTS_DIR = Path(globals().get("OUTPUTS_DIR", "./outputs"))
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Clear in-memory globals that can conflict with fresh builds
for v in ["rows_top5", "conf_map", "CONF_HINTS"]:
    if v in globals():
        del globals()[v]

# Remove zero-byte artifacts that break pandas read_csv
for name in ["role_confidence_top5.csv","role_confidence_top5.json","role_confidence_indicators.csv"]:
    p = OUTPUTS_DIR / name
    try:
        if p.exists() and p.stat().st_size == 0:
            p.unlink()
            print("[15.19] Removed empty:", p.name)
    except Exception as e:
        print("[15.19] Could not inspect/remove", p.name, e)

print("[15.19] Confidence state reset.")


[15.19] Confidence state reset.


In [15]:
# ===== Cell 15.2 — Role Confidence Shortlist (robust build + canonicalize + write; empty-safe) =====
import os, io, re, json
import pandas as pd
from pathlib import Path

OUTPUTS_DIR = Path(globals().get("OUTPUTS_DIR", "./outputs"))
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# --- canonicalization (keep consistent with 16.97/16.979) ---
VALID_ROLES = set(globals().get("VALID_ROLES", [])) or {
    "Director","Manager","Supervisor","Specialist","Analyst","Technician","Advisor",
    "Teacher","Coach","Liaison","Architect","Designer","Principal","Executive Director",
    "Lead Tech","Coordinator","Accountant","Partner","Assistant Principal","Translator"
}
LEVEL_TOKS = {"lead","i","ii","iii"}
ROLE_SYNONYMS = {
    "instructor": "Teacher",
    "exec dir": "Executive Director",
    "executive dir": "Executive Director",
    "ap": "Assistant Principal",
}
def _canon_role(s: str) -> str:
    s = (s or "").strip()
    low = s.lower()
    parts = [p for p in re.split(r"[ /\-]+", low) if p]
    parts = [p for p in parts if p not in LEVEL_TOKS]  # strip leaked levels
    low2 = " ".join(parts)
    for k,v in ROLE_SYNONYMS.items():
        if k in low2: return v
    for vr in VALID_ROLES:
        if re.fullmatch(rf"{re.escape(vr)}", s, flags=re.I): return vr
    for vr in VALID_ROLES:
        if re.match(rf"^{re.escape(vr)}\b", s, flags=re.I): return vr
    return " ".join(w.capitalize() for w in low2.split())

# --- robust table reader ---
def _read_table_robust(path: Path):
    if not path or not path.exists() or path.stat().st_size == 0:
        return None
    encs = ["utf-8","utf-8-sig","cp1252","latin1","windows-1252","utf-16","utf-16le","utf-16be"]
    seps = [None, ",", "\t", ";", "|"]
    for enc in encs:
        for sep in seps:
            try:
                df = pd.read_csv(path, encoding=enc, sep=sep, engine="python")
                if df.shape[1] >= 1:
                    return df
            except Exception:
                pass
    try:
        txt = path.read_text(errors="ignore")
        return pd.read_csv(io.StringIO(txt), engine="python")
    except Exception:
        return None

# --- artifact paths ---
conf_long = OUTPUTS_DIR / "role_confidence_indicators.csv"
conf_top5 = OUTPUTS_DIR / "role_confidence_top5.csv"
conf_json = OUTPUTS_DIR / "role_confidence_top5.json"

# --- try to use existing in-memory results first ---
rows_top5 = globals().get("rows_top5", None)
conf_map  = globals().get("conf_map", None)

def _rebuild_from_json():
    if not conf_json.exists() or conf_json.stat().st_size == 0: return None, None
    try:
        raw = json.loads(conf_json.read_text(encoding="utf-8"))
    except Exception:
        return None, None
    rows, cmap = [], {}
    for k, lst in (raw or {}).items():
        if not str(k).isdigit(): continue
        rid = int(k)
        norm = []
        for d in (lst or [])[:5]:
            role = _canon_role(d.get("role",""))
            conf = d.get("confidence", None)
            try: conf = float(conf) if conf not in [None, ""] else None
            except Exception: conf = None
            if role:
                norm.append((role, conf))
                rows.append({"row_id": rid, "role": role, "confidence": conf})
        cmap[rid] = norm
    return rows, cmap

def _rebuild_from_top5_csv():
    if not conf_top5.exists() or conf_top5.stat().st_size == 0: return None, None
    df = _read_table_robust(conf_top5)
    if df is None or df.empty: return None, None
    cols = {c.lower(): c for c in df.columns}
    rid = cols.get("row_id"); role = cols.get("role"); conf = cols.get("confidence")
    if not (rid and role): return None, None
    rows, cmap = [], {}
    for _, r in df.iterrows():
        try: rr = int(r[rid])
        except Exception: continue
        ro = _canon_role(r[role])
        try: cf = float(r[conf]) if conf and pd.notna(r[conf]) else None
        except Exception: cf = None
        rows.append({"row_id": rr, "role": ro, "confidence": cf})
        cmap.setdefault(rr, [])
        if len(cmap[rr]) < 5: cmap[rr].append((ro, cf))
    return rows, cmap

def _rebuild_from_long_csv():
    if not conf_long.exists() or conf_long.stat().st_size == 0: return None, None
    df = _read_table_robust(conf_long)
    if df is None or df.empty: return None, None
    cols = {c.lower(): c for c in df.columns}
    rid = cols.get("row_id"); role = cols.get("role"); conf = cols.get("confidence")
    if not (rid and role): return None, None
    if conf and conf in df:
        df["_conf"] = pd.to_numeric(df[conf], errors="coerce")
        df = df.sort_values(["row_id","_conf"], ascending=[True, False])
    rows, cmap = [], {}
    for rid_val, grp in df.groupby(df[rid]):
        try: rr = int(rid_val)
        except Exception: continue
        take = grp.head(5)
        lst = []
        for _, r in take.iterrows():
            ro = _canon_role(r[role])
            cf = None
            if conf and conf in r:
                try: cf = float(r[conf]) if pd.notna(r[conf]) else None
                except Exception: cf = None
            rows.append({"row_id": rr, "role": ro, "confidence": cf})
            lst.append((ro, cf))
        cmap[rr] = lst
    return rows, cmap

# --- rebuild if not provided in-memory ---
if rows_top5 is None or conf_map is None:
    rows_top5, conf_map = _rebuild_from_json()
if rows_top5 is None or conf_map is None:
    rows_top5, conf_map = _rebuild_from_top5_csv()
if rows_top5 is None or conf_map is None:
    rows_top5, conf_map = _rebuild_from_long_csv()

# --- if still empty, create safe placeholders ---
if rows_top5 is None or conf_map is None:
    rows_top5, conf_map = [], {}
    print("[15.2] Warning: no confidence data found; writing empty shortlist artifacts.")

# --- idempotent canonicalization & cleanup ---
rows_top5 = [
    {"row_id": int(rec.get("row_id", -1)),
     "role": _canon_role(rec.get("role","")),
     "confidence": (None if rec.get("confidence", None) in ["", None] else float(rec.get("confidence")))}
    for rec in rows_top5
]
conf_map = {
    int(k): [(_canon_role(r), (None if c in ["", None] else float(c))) for (r,c) in v]
    for k,v in conf_map.items()
}

# --- materialize DataFrames (empty-safe with headers) ---
df_long  = pd.DataFrame([{"row_id": int(k), "role": ro, "confidence": cf}
                         for k, lst in conf_map.items() for (ro, cf) in lst],
                        columns=["row_id","role","confidence"])
df_top5  = pd.DataFrame(rows_top5, columns=["row_id","role","confidence"])

# --- write artifacts ---
df_long.to_csv(conf_long, index=False, encoding="utf-8")
df_top5.to_csv(conf_top5, index=False, encoding="utf-8")

# JSON: handle empty safely
conf_out = {}
if not df_top5.empty:
    for rid, grp in df_top5.groupby("row_id"):
        lst = []
        for _, r in grp.head(5).iterrows():
            lst.append({"role": r["role"], "confidence": (None if pd.isna(r.get("confidence")) else float(r["confidence"]))})
        conf_out[str(int(rid))] = lst
conf_json.write_text(json.dumps(conf_out, ensure_ascii=False, indent=2), encoding="utf-8")

# expose to globals for 15.201 / 15.351
globals()["rows_top5"] = rows_top5
globals()["conf_map"]  = conf_map

print(f"[15.2] Saved -> {conf_long.name} ({conf_long.stat().st_size} bytes)")
print(f"[15.2] Saved -> {conf_top5.name} ({conf_top5.stat().st_size} bytes)")
print(f"[15.2] Saved -> {conf_json.name} ({conf_json.stat().st_size} bytes)")
print(f"[15.2] Shortlist rows: {len(rows_top5)} | entries in map: {len(conf_map)}")


[15.2] Warning: no confidence data found; writing empty shortlist artifacts.
[15.2] Saved -> role_confidence_indicators.csv (23 bytes)
[15.2] Saved -> role_confidence_top5.csv (23 bytes)
[15.2] Saved -> role_confidence_top5.json (2 bytes)
[15.2] Shortlist rows: 0 | entries in map: 0


In [16]:
# ===== Cell 15.201 — Load CONF_HINTS for injection into Call #2 =====
import json, pandas as pd
from pathlib import Path
OUTPUTS_DIR = Path(globals().get("OUTPUTS_DIR","./outputs"))

conf_json = OUTPUTS_DIR / "role_confidence_top5.json"
conf_csv  = OUTPUTS_DIR / "role_confidence_top5.csv"
CONF_HINTS = {}

def _canon_role(s: str):
    # lite canonicalization (must match what you use downstream)
    s = (s or "").strip()
    s2 = s.lower()
    if "instructor" in s2: return "Teacher"
    if "exec dir" in s2 or "executive dir" in s2: return "Executive Director"
    return " ".join(w.capitalize() for w in s2.split())

if conf_json.exists() and conf_json.stat().st_size > 0:
    raw = json.loads(conf_json.read_text(encoding="utf-8"))
    for k, lst in raw.items():
        try: rid = int(k)
        except: continue
        CONF_HINTS[rid] = [(_canon_role(d.get("role","")), d.get("confidence", None)) for d in (lst or [])]

elif conf_csv.exists() and conf_csv.stat().st_size > 0:
    df = pd.read_csv(conf_csv)
    for rid, grp in df.groupby("row_id"):
        try: rid_i = int(rid)
        except: continue
        CONF_HINTS[rid_i] = [(_canon_role(r["role"]), r.get("confidence", None)) for _, r in grp.head(5).iterrows()]

globals()["CONF_HINTS"] = CONF_HINTS
print(f"[15.201] Loaded CONF_HINTS for {len(CONF_HINTS)} rows (now available to injector).")


[15.201] Loaded CONF_HINTS for 0 rows (now available to injector).


In [17]:
# ===== Cell 15.205 — Post-15.2 sanity =====
import json, pandas as pd
from pathlib import Path

OUTPUTS_DIR = Path(globals().get("OUTPUTS_DIR", "./outputs"))
top5_csv  = OUTPUTS_DIR / "role_confidence_top5.csv"
top5_json = OUTPUTS_DIR / "role_confidence_top5.json"
long_csv  = OUTPUTS_DIR / "role_confidence_indicators.csv"

# Globals present?
assert "rows_top5" in globals(), "rows_top5 missing — 15.2 did not define it."
assert "conf_map"  in globals(), "conf_map missing — 15.2 did not define it."
print(f"[15.205] rows_top5 entries: {len(rows_top5)} | conf_map keys: {len(conf_map)}")

# Artifacts present?
assert top5_csv.exists(),  f"{top5_csv.name} missing"
assert top5_json.exists(), f"{top5_json.name} missing"
assert long_csv.exists(),  f"{long_csv.name} missing"

# Read top5 CSV
df = pd.read_csv(top5_csv, engine="python")
print(f"[15.205] {top5_csv.name}: rows={len(df)} cols={len(df.columns)}")

# Read JSON
raw = json.loads(top5_json.read_text(encoding="utf-8"))
print(f"[15.205] {top5_json.name}: entries={len(raw)}")

print("[15.205] Confidence shortlist looks good.")


[15.205] rows_top5 entries: 0 | conf_map keys: 0
[15.205] role_confidence_top5.csv: rows=0 cols=3
[15.205] role_confidence_top5.json: entries=0
[15.205] Confidence shortlist looks good.


In [18]:
# ===== Cell 15.21 — Confidence artifacts sanity check (defines ROLE_CONF_* & inspects) =====
from pathlib import Path
import pandas as pd, json, io

OUTPUTS_DIR = Path(globals().get("OUTPUTS_DIR", "./outputs"))
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Define the canonical paths & export to globals for other cells
ROLE_CONF_CSV_LONG = str(OUTPUTS_DIR / "role_confidence_indicators.csv")
ROLE_CONF_CSV_TOP5 = str(OUTPUTS_DIR / "role_confidence_top5.csv")
ROLE_CONF_JSON     = str(OUTPUTS_DIR / "role_confidence_top5.json")
globals().update({
    "ROLE_CONF_CSV_LONG": ROLE_CONF_CSV_LONG,
    "ROLE_CONF_CSV_TOP5": ROLE_CONF_CSV_TOP5,
    "ROLE_CONF_JSON": ROLE_CONF_JSON,
})

def _read_table_robust(p: Path):
    encs = ["utf-8","utf-8-sig","cp1252","latin1","windows-1252","utf-16","utf-16le","utf-16be"]
    seps = [None, ",", "\t", ";", "|"]  # None = sniff
    for enc in encs:
        for sep in seps:
            try:
                df = pd.read_csv(p, encoding=enc, sep=sep, engine="python")
                if df.shape[1] >= 1:
                    return df
            except Exception:
                pass
    try:
        txt = p.read_text(errors="ignore")
        return pd.read_csv(io.StringIO(txt), engine="python")
    except Exception:
        return None

checks = [
    ("role_confidence_indicators.csv", Path(ROLE_CONF_CSV_LONG)),
    ("role_confidence_top5.csv",       Path(ROLE_CONF_CSV_TOP5)),
    ("role_confidence_top5.json",      Path(ROLE_CONF_JSON)),
]

for name, p in checks:
    exists = p.exists()
    size = p.stat().st_size if exists else 0
    extra = ""
    if exists and name.endswith(".csv"):
        df = _read_table_robust(p)
        extra = (f" | rows={len(df)} cols={len(df.columns)}" if (df is not None and not df.empty) else " | rows=0 cols=0")
    if exists and name.endswith(".json"):
        try:
            data = json.loads(p.read_text(encoding="utf-8"))
            extra = f" | entries={len(data)}"
        except Exception as e:
            extra = f" | json error: {e}"
    print(f"{name}: exists={exists} size={size}{extra}")

# Friendly hint if everything is present but empty
all_present = all(p.exists() for _, p in checks)
any_nonempty = any(p.exists() and p.stat().st_size > 0 for _, p in checks)
if all_present and not any_nonempty:
    print("[15.21] Found artifacts but they are empty — run your confidence generator (first API call) before 15.2.")


role_confidence_indicators.csv: exists=True size=23 | rows=0 cols=0
role_confidence_top5.csv: exists=True size=23 | rows=0 cols=0
role_confidence_top5.json: exists=True size=2 | entries=0


In [19]:
# ===== Cell 15.25 — Backfill confidence JSON from Top-5 CSV (canonical roles updated) =====
from pathlib import Path
import pandas as pd, json, math

OUTPUTS_DIR = Path(globals().get("OUTPUTS_DIR", "./outputs"))
conf_top5 = OUTPUTS_DIR / "role_confidence_top5.csv"
conf_json = OUTPUTS_DIR / "role_confidence_top5.json"

VALID_ROLES = globals().get("VALID_ROLES", [])
ROLE_SYNONYMS = globals().get("ROLE_SYNONYMS", {})

def canonical_role(s: str) -> str:
    if not isinstance(s, str): s = str(s or "")
    raw = s.strip(); low = raw.lower()
    # synonyms
    for k, v in ROLE_SYNONYMS.items():
        if k in low:
            return v
    # exact
    for r in VALID_ROLES:
        if low == r.lower(): return r
    # substring
    for r in sorted(VALID_ROLES, key=lambda x: -len(x)):
        if r.lower() in low or low in r.lower(): return r
    # keep Athletic Coach as-is (used only for disambiguation signals)
    if "athletic coach" in low: return "Athletic Coach"
    return raw.strip().title()

def _isnum(x):
    try: return not math.isnan(float(x))
    except Exception: return False

if not conf_top5.exists():
    raise FileNotFoundError(f"Top-5 CSV not found: {conf_top5}")

df = pd.read_csv(conf_top5)
conf_map = {}
for idx, r in df.iterrows():
    rid = r.get("row_id", idx)
    try: rid = int(rid)
    except Exception: rid = int(idx)
    entries = []
    for i in range(1, 6):
        role = canonical_role(r.get(f"top{i}_role", ""))
        conf = r.get(f"top{i}_confidence", None)
        conf = float(conf) if _isnum(conf) else None
        if role:
            entries.append({"role": role, "confidence": conf, "rationale": ""})
    conf_map[rid] = entries

conf_json.write_text(json.dumps(conf_map, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"[Backfill] Wrote JSON map -> {conf_json} (records={len(conf_map)})")


[Backfill] Wrote JSON map -> /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251001_165753/outputs/role_confidence_top5.json (records=0)


In [20]:
USE_CONF_SHORTLIST_IN_PROMPT = True   # turn injection on/off
CONF_K = 3                            # shortlist size
CONF_MIN = 0.20                       # ignore roles below this confidence
ENFORCE_MIN_BOUNDS_TEXT = True        # keep HARD FILTER text in prompt


In [21]:
# ===== Cell 15.35 — Confidence injection + leadership/coach disambiguation =====
from pathlib import Path

USE_CONF_SHORTLIST_IN_PROMPT = globals().get("USE_CONF_SHORTLIST_IN_PROMPT", True)
CONF_K = int(globals().get("CONF_K", 3))
CONF_MIN = float(globals().get("CONF_MIN", 0.20))
ENFORCE_MIN_BOUNDS_TEXT = bool(globals().get("ENFORCE_MIN_BOUNDS_TEXT", True))

ROLE_SYNONYMS = globals().get("ROLE_SYNONYMS", {})
VALID_ROLES = globals().get("VALID_ROLES", [])

LEADERSHIP_TOKENS = globals().get("LEADERSHIP_TOKENS", set())
AP_TOKENS = globals().get("AP_TOKENS", set())
PRINCIPAL_TOKENS = globals().get("PRINCIPAL_TOKENS", set())
COACH_TOKENS = globals().get("COACH_TOKENS", set())
ATHLETIC_TOKENS = globals().get("ATHLETIC_TOKENS", set())

def _row_get(row, key, default=""):
    try:
        if hasattr(row, "get"): v = row.get(key, default)
        else: v = row[key] if key in row else default
        return "" if v is None else str(v)
    except Exception:
        return default

def _text_blob(row):
    parts = [
        _row_get(row, "Position Summary"),
        _row_get(row, "Essential Functions"),
        _row_get(row, "Knowledge, Skills and Abilities"),
        _row_get(row, "Education"),
        _row_get(row, "Work Experience"),
        _row_get(row, "Licenses and Certifications"),
    ]
    return " ".join(p for p in parts if p).lower()

def _has_any(txt, vocab): return any(tok in txt for tok in vocab)

def _leadership_coach_block(row):
    txt = _text_blob(row)

    hit_lead = _has_any(txt, LEADERSHIP_TOKENS)
    hit_ap   = _has_any(txt, AP_TOKENS)
    hit_prin = _has_any(txt, PRINCIPAL_TOKENS)
    hit_ic   = _has_any(txt, COACH_TOKENS)
    hit_ath  = _has_any(txt, ATHLETIC_TOKENS)

    lines = []
    if any([hit_lead, hit_ap, hit_prin, hit_ic, hit_ath]):
        lines.append("### Leadership & Instructional Coach Disambiguation")
        if hit_prin or hit_ap or hit_lead:
            lines.append("- Signals indicate school leadership responsibilities (Principal / Assistant Principal).")
            lines.append("  *Eligibility minima for Principal/AP:* administrator/leadership credential (or explicit requirement),")
            lines.append("  plus required years of teaching/leadership experience as stated.")
            lines.append("  If both Principal and Assistant Principal are plausible, choose the role that best matches the")
            lines.append("  scope of authority (final evaluator/budgetary authority → Principal; delegated discipline/testing/ops → Assistant Principal).")
        if hit_ic:
            lines.append("- Signals indicate *instructional coaching* (non-athletic): PD/PLCs, modeling lessons, coaching cycles, curriculum/data support.")
            lines.append("  *Eligibility minima for Coach (instructional):* active/previous teaching license and classroom experience as required.")
        if hit_ath:
            lines.append("- Athletic/sports terms detected. Exclude *Coach (instructional)* if duties are athletic-only.")
    return "\n".join(lines)

def _hint_pairs_for_row(row):
    if not USE_CONF_SHORTLIST_IN_PROMPT: return []
    conf_by_id = globals().get("CONF_HINTS", {})
    conf_by_nm = globals().get("CONF_HINTS_BY_NAME", {})

    # prefer row_id
    rid = None
    try:
        rid = int(row.get("row_id", getattr(row, "name", 0)))
    except Exception:
        rid = None
    pairs = conf_by_id.get(rid, []) if rid is not None else []

    # fallback by job name
    if not pairs:
        nm = _row_get(row, "Job Description Name").strip().casefold()
        if nm: pairs = conf_by_nm.get(nm, [])

    # filter + truncate
    cleaned = []
    for role, conf in pairs:
        if not role: continue
        try: conf = float(conf) if conf is not None else None
        except Exception: conf = None
        if conf is None or conf >= CONF_MIN:
            cleaned.append((role, conf))
    return cleaned[:CONF_K]

def _hint_block_for_row(row):
    pairs = _hint_pairs_for_row(row)
    extra = _leadership_coach_block(row)
    if not pairs and not extra:
        return ""
    lines = []
    if pairs:
        lines.append("### Confidence Shortlist (from cross-resource comparison)")
        for role, conf in pairs:
            if conf is None: lines.append(f"- {role}")
            else:
                try: lines.append(f"- {role} (confidence {conf:.2f})")
                except Exception: lines.append(f"- {role} (confidence {conf})")
    if ENFORCE_MIN_BOUNDS_TEXT:
        lines += [
            "",
            "**Minimum bounds policy (HARD FILTER):**",
            "- A role is INELIGIBLE if minimum Education, required Licenses/Certifications,",
            "  or minimum Work Experience are not met for that role. Do not select ineligible roles.",
            "- If multiple eligible roles remain, prefer the highest-confidence role from the shortlist",
            "  that best aligns to KSACs and essential functions.",
        ]
    if extra:
        lines.append("")
        lines.append(extra)
    return "\n".join(lines)

# Establish a base builder if not already present
def _build_base_prompt_from_row(row):
    zsp = globals().get("zero_shot_prompt", "").strip()
    if not zsp:
        zsp = ("Objective: Classify into a valid MNPS major role group and level (Lead/I/II/III) using KSACs "
               "and eligibility minima, focusing on duties and competencies.")
    def g(k): return _row_get(row, k, "")
    blocks = [
        zsp,
        f"Job Description Name: {g('Job Description Name')}",
        f"Position Summary: {g('Position Summary')}",
        f"Essential Functions: {g('Essential Functions')}",
        f"Knowledge, Skills and Abilities: {g('Knowledge, Skills and Abilities')}",
        f"Education: {g('Education')}",
        f"Work Experience: {g('Work Experience')}",
        f"Licenses and Certifications: {g('Licenses and Certifications')}",
    ]
    return "\n".join(blocks).strip()

if 'full_text_for_row' in globals():
    _ORIG_full_text_for_row = full_text_for_row
else:
    if 'build_prompt' in globals():
        def _ORIG_full_text_for_row(row):
            try:
                prompt, _ = build_prompt(row); return str(prompt)
            except Exception:
                return _build_base_prompt_from_row(row)
    else:
        def _ORIG_full_text_for_row(row):
            return _build_base_prompt_from_row(row)

def full_text_for_row(row):
    base = _ORIG_full_text_for_row(row)
    addon = _hint_block_for_row(row)
    return (f"{base}\n\n{addon}\n") if addon else base

print("[15.35] Injector ready — shortlist + leadership/coach disambiguation + minima appended.")


[15.35] Injector ready — shortlist + leadership/coach disambiguation + minima appended.


In [22]:
# ===== Cell 15.351 — No-Title injector + shortlist + presence guards =====
import re, math, json
import pandas as pd

# knobs
USE_CONF_SHORTLIST_IN_PROMPT = True
CONF_K   = int(globals().get("CONF_K", 3))
CONF_MIN = float(globals().get("CONF_MIN", 0.20))
STRICT_SHORTLIST = bool(globals().get("STRICT_SHORTLIST", False))   # set True to force choice from shortlist unless ineligible
ENFORCE_MIN_BOUNDS_TEXT = True
STRICT_NO_TITLE  = True

# load shortlist maps if available (populated by Cell 15.2 / 15.25)
CONF_HINTS = globals().get("CONF_HINTS", {})        # {row_id: [(role, conf), ...]}
# IMPORTANT: disable name-based hints — prevents title leakage
CONF_HINTS_BY_NAME = {}

def _to_str(v):
    try:
        if v is None or (isinstance(v, float) and math.isnan(v)) or pd.isna(v):
            return ""
    except Exception:
        pass
    return str(v) if v is not None else ""

def _get(row, key):
    try:
        if hasattr(row, "get"):
            return _to_str(row.get(key, ""))
        return _to_str(row[key]) if key in row else ""
    except Exception:
        return ""

def _len_ok(s, n=15): return len((s or "").strip()) >= n
def _blob(row):
    # NO title here — only determinant factors
    return " ".join([
        _get(row, "Position Summary"),
        _get(row, "Essential Functions"),
        _get(row, "Knowledge, Skills and Abilities"),
        _get(row, "Education"),
        _get(row, "Work Experience"),
        _get(row, "Licenses and Certifications"),
    ])

def _shortlist_pairs(row):
    rid = None
    try: rid = int(row.get("row_id", getattr(row, "name", 0)))
    except Exception: rid = None
    pairs = CONF_HINTS.get(rid, []) if rid is not None else []
    out = []
    for role, conf in pairs:
        try: conf = float(conf) if conf is not None else None
        except Exception: conf = None
        if role and (conf is None or conf >= CONF_MIN): out.append((role, conf))
    return out[:CONF_K]

def _hint_block(row):
    blocks = []
    # dynamic presence guards — prevent “lack of detailed information” claims when fields are present
    ps, ef, ks, ed, xp, lc = (
        _get(row,'Position Summary'), _get(row,'Essential Functions'), _get(row,'Knowledge, Skills and Abilities'),
        _get(row,'Education'), _get(row,'Work Experience'), _get(row,'Licenses and Certifications')
    )
    guards = ["**You must NOT claim that any field is missing if it is present below.**"]
    if _len_ok(ps): guards.append("- Position Summary is provided; do not say it is missing.")
    if _len_ok(ef): guards.append("- Essential Functions are provided; do not say they are missing.")
    if _len_ok(ks): guards.append("- KSAs are provided; do not say they are missing.")
    if _len_ok(ed): guards.append("- Education is provided; do not say it is missing.")
    if _len_ok(xp): guards.append("- Work Experience is provided; do not say it is missing.")
    if _len_ok(lc): guards.append("- Licenses/Certifications are provided; do not say they are missing.")
    blocks += guards

    pairs = _shortlist_pairs(row)
    if pairs:
        blocks.append("\n### Confidence Shortlist (from KSAC comparison — title excluded)")
        for role, conf in pairs:
            blocks.append(f"- {role}" + ("" if conf is None else f" (confidence {conf:.2f})"))
        if STRICT_SHORTLIST:
            blocks.append("\nChoose from this shortlist unless all options clearly fail eligibility minima. "
                          "If you deviate, begin explanation with 'DEVIATION:' and explain why.")

    if ENFORCE_MIN_BOUNDS_TEXT:
        blocks += ["",
            "**Eligibility minima (HARD FILTER):**",
            "- INELIGIBLE if minimum Education, required Licenses/Certifications, or minimum Experience are not met.",
        ]

    if STRICT_NO_TITLE:
        blocks += ["",
            "**Do NOT use or reference the job title in any way.**",
            "- Base the decision solely on: Position Summary, Essential Functions, KSAs, Education, Work Experience, and Licenses/Certifications.",
            "- Never justify with 'based on the job title' or similar phrasing.",
        ]
    return "\n".join(blocks).strip()

def _base_prompt(row):
    zsp = (globals().get("zero_shot_prompt","") or "").strip() or \
          "Objective: Classify into a valid MNPS major role group and level (Lead/I/II/III) using KSACs and eligibility minima."
    return "\n".join([
        zsp,
        f"Position Summary: {_get(row,'Position Summary')}",
        f"Essential Functions: {_get(row,'Essential Functions')}",
        f"Knowledge, Skills and Abilities: {_get(row,'Knowledge, Skills and Abilities')}",
        f"Education: {_get(row,'Education')}",
        f"Work Experience: {_get(row,'Work Experience')}",
        f"Licenses and Certifications: {_get(row,'Licenses and Certifications')}",
    ]).strip()

def full_text_for_row(row):
    base = _base_prompt(row)
    hints = _hint_block(row)
    return base + ("\n\n" + hints if hints else "")

globals()["BASE_full_text_for_row"] = full_text_for_row
print("[15.351] Installed: no-title injector + shortlist + field-presence guards")


[15.351] Installed: no-title injector + shortlist + field-presence guards


In [23]:
# ===== Cell 15.36 — Prompt auditor (assert NO TITLE, log field lengths; BATCH_INPUT_CSV optional) =====
import pandas as pd, io, re
from pathlib import Path

# --- require injector so we audit the same prompt builder used in batch ---
if "full_text_for_row" not in globals():
    raise RuntimeError("Prompt injector missing. Run Cell 15.351 first so 'full_text_for_row' is defined.")

# --- robust readers & auto-finder ---
def _read_csv_robust(path: str) -> pd.DataFrame:
    encs = ["utf-8","utf-8-sig","cp1252","latin1","windows-1252","utf-16","utf-16le","utf-16be"]
    seps = [None, ",", "\t", ";", "|"]  # None = sniff
    for enc in encs:
        for sep in seps:
            try:
                df = pd.read_csv(path, encoding=enc, sep=sep, engine="python")
                if df.shape[1] >= 1:
                    return df
            except Exception:
                pass
    # last-chance: manual decode + sniff
    raw = Path(path).read_bytes()
    for enc in encs:
        try:
            txt = raw.decode(enc, errors="ignore")
            for sep in seps:
                try:
                    df = pd.read_csv(io.StringIO(txt), sep=sep, engine="python")
                    if df.shape[1] >= 1:
                        return df
                except Exception:
                    pass
        except Exception:
            pass
    raise ValueError(f"[15.36] Could not parse CSV: {path}")

def _auto_find_input():
    names = ["Sample JDs", "New Sample", "Sample_JDs", "Sample", "JDs"]
    roots = [
        Path(globals().get("INPUTS_DIR",".")),
        Path.cwd(),
        Path("/content"),
        Path("/content/drive/My Drive/Colab Notebooks"),
        Path("/content/drive/My Drive/Colab Notebooks/Data Inputs"),
    ]
    cands = []
    for r in roots:
        if not r.exists():
            continue
        try:
            for p in r.rglob("*.csv"):
                if any(n.lower() in p.name.lower() for n in names):
                    cands.append(p)
        except Exception:
            pass
    if not cands:
        return None
    cands.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return str(cands[0])

# --- obtain the input dataframe (no hard requirement on BATCH_INPUT_CSV) ---
if "in_df" in globals() and isinstance(in_df, pd.DataFrame) and len(in_df):
    src_msg = "[15.36] Using existing 'in_df'."
    _df = in_df.copy()
elif "df" in globals() and isinstance(df, pd.DataFrame) and len(df):
    src_msg = "[15.36] Using existing 'df' as input."
    _df = df.copy()
elif "BATCH_INPUT_CSV" in globals() and BATCH_INPUT_CSV and Path(BATCH_INPUT_CSV).exists():
    src_msg = f"[15.36] Loaded from BATCH_INPUT_CSV: {BATCH_INPUT_CSV}"
    _df = _read_csv_robust(BATCH_INPUT_CSV)
else:
    guess = _auto_find_input()
    if guess:
        src_msg = f"[15.36] Auto-found input CSV: {guess}"
        _df = _read_csv_robust(guess)
    else:
        raise RuntimeError("[15.36] Could not locate input data — set BATCH_INPUT_CSV or run your Inputs/Load cell.")

print(src_msg)

# --- ensure row_id exists for alignment with shortlist & predictions ---
if "row_id" not in _df.columns:
    _df = _df.reset_index().rename(columns={"index":"row_id"})

# --- audit prompts: title absence + determinant lengths ---
rows = []
for _, row in _df.iterrows():
    title = str(row.get("Job Description Name","") or "")
    txt = full_text_for_row(row)
    contains_title = bool(title) and (title.lower() in txt.lower())
    rows.append({
        "row_id": int(row["row_id"]),
        "title_in_prompt": contains_title,
        "len_ps": len(str(row.get("Position Summary",""))),
        "len_ef": len(str(row.get("Essential Functions",""))),
        "len_ks": len(str(row.get("Knowledge, Skills and Abilities",""))),
        "len_ed": len(str(row.get("Education",""))),
        "len_xp": len(str(row.get("Work Experience",""))),
        "len_lc": len(str(row.get("Licenses and Certifications",""))),
    })

audit = pd.DataFrame(rows, columns=[
    "row_id","title_in_prompt","len_ps","len_ef","len_ks","len_ed","len_xp","len_lc"
])
out = Path(globals().get("OUTPUTS_DIR","./outputs")) / "prompt_audit.csv"
out.parent.mkdir(parents=True, exist_ok=True)
audit.to_csv(out, index=False, encoding="utf-8")
print("[15.36] Wrote prompt audit ->", out)

if audit["title_in_prompt"].any():
    bad = audit[audit["title_in_prompt"]].head(10)
    print("WARNING: Some prompts still contain the title. Showing few offenders:")
    try:
        display(bad)
    except Exception:
        print(bad.to_string(index=False))
else:
    print("OK: No prompts contain the job title.")


[15.36] Auto-found input CSV: /content/drive/MyDrive/Colab Notebooks/Run Results/RUN_20250925_193216/outputs/disagreements_sample.csv
[15.36] Wrote prompt audit -> /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251001_165753/outputs/prompt_audit.csv
OK: No prompts contain the job title.


Create classifications using OpenAI. Of note here is:
* The **developer** prompt - this is the "system prompt" or "custom instructions" for the model. This determines the overall behavior of the model.
* The **user** prompt - this is what we send to the model like when we're chatting with ChatGPT.

# **Version 6.0**
> Responses API (TEXT-ONLY, no attachments), saves to OUTPUTS_DIR  
> Composes text-only input (no attachments)  
> Newer SDK: server-enforced  
> Structured Outputs via parse; uses jsons  
> Save outputs to OUTPUTS_DIR


In [24]:
# ===== Cell 16 — Responses API (TEXT-ONLY, no attachments), saves to OUTPUTS_DIR =====
from openai import OpenAI
from pathlib import Path
import pandas as pd, json, inspect

client = OpenAI()  # API key from Environment Setup

# ---- Requires earlier cells ----
assert 'MODEL_ID' in globals(), "Run Environment Setup first (MODEL_ID)."
assert 'INPUTS_DIR' in globals() and 'OUTPUTS_DIR' in globals(), "Run the unique-run Cell 4 first."
assert 'zero_shot_prompt' in globals(), "Define zero_shot_prompt in your prompt cell."
assert 'job_desc_text' in globals(), "Cell 4 builds job_desc_text (row 0 smoke test)."

print("🤖 Using model:", MODEL_ID)

# If read_csv_smart exists (Cell 4), use it for robust encodings; else default to utf-8
def _read_csv(path: Path, **kw):
    if 'read_csv_smart' in globals():
        return read_csv_smart(path, **kw)
    return pd.read_csv(path, encoding="utf-8", **kw)

# ---- Build a SMALL context from Ground Truth (first 3 rows) ----
gt_path = Path(INPUTS_DIR) / "Ground Truth Masterfile.csv"
context_block = ""
if gt_path.exists():
    try:
        gt_df = _read_csv(gt_path).fillna("")
        # keep only lightweight columns if present
        preferred_cols = [
            "Original Job Title","New Job Title","Major Role Group","Minor Sub-Group","Justification for Grouping",
            "Position Summary","Education","Work Experience","Licenses and Certifications","Essential Functions","Knowledge, Skills and Abilities"
        ]
        cols = [c for c in preferred_cols if c in gt_df.columns] or list(gt_df.columns)[:8]
        mini = gt_df[cols].head(3)
        # represent as compact JSON so the model can parse easily
        context_block = "Context (Ground Truth examples):\n" + mini.to_json(orient="records", force_ascii=False)
    except Exception as e:
        context_block = f"Context note: Ground Truth CSV present but could not be summarized ({e})."

# ---- Compose text-only input (no attachments) ----
# Tip: the zero_shot_prompt you wrote mentions "attached reference sources" — we add a Context block instead.
full_text = (
    zero_shot_prompt.strip()
    + "\n\n"
    + (context_block + "\n\n" if context_block else "")
    + "Classify the following job description:\n\n"
    + job_desc_text
)

# ---- Capability detection for your SDK version ----
def _has_param(obj, name: str) -> bool:
    try:
        return name in inspect.signature(obj).parameters
    except Exception:
        return False

supports_parse_schema  = _has_param(client.responses.parse,  "response_format")
supports_create_schema = _has_param(client.responses.create, "response_format")

parsed = None
raw_text = ""

try:
    if supports_parse_schema:
        # Newer SDK: server-enforced Structured Outputs via parse()
        resp = client.responses.parse(
            model=MODEL_ID,
            input=[{"role": "user", "content": [{"type":"input_text","text": full_text}]}],
            temperature=0.2,
            max_output_tokens=1400,
            response_format=JobClassificationTable,  # Pydantic schema (Cells 14–15)
        )
        parsed  = resp.output_parsed
        raw_text = resp.output_text or ""
    elif supports_create_schema:
        # Mid SDK: enforce via create() + json_schema
        schema = JobClassificationTable.model_json_schema()
        resp = client.responses.create(
            model=MODEL_ID,
            input=[{"role": "user", "content": [{"type":"input_text","text": full_text}]}],
            temperature=0.2,
            max_output_tokens=1400,
            response_format={
                "type": "json_schema",
                "json_schema": {"name": "JobClassificationTable", "schema": schema, "strict": True},
            },
        )
        raw_text = getattr(resp, "output_text", None) or ""
        # Clean the raw_text to remove potential markdown formatting
        if raw_text.strip().startswith("```json"):
            raw_text = raw_text.strip()[7:].strip("`")
        parsed = JobClassificationTable.model_validate_json(raw_text) if raw_text else None
    else:
        # Old SDK: prompt-only enforcement + client-side validation
        schema_json = json.dumps(JobClassificationTable.model_json_schema(), indent=2)
        strict_text = (
            "You MUST return ONLY valid JSON that matches the following JSON Schema. No prose, no markdown.\n"
            "JSON Schema:\n" + schema_json + "\n\n"
            "Task:\n" + full_text
        )
        resp = client.responses.create(
            model=MODEL_ID,
            input=[{"role": "user", "content": [{"type":"input_text","text": strict_text}]}],
            temperature=0.2,
            max_output_tokens=1400,
        )
        raw_text = getattr(resp, "output_text", None) or ""
        # Clean the raw_text to remove potential markdown formatting
        if raw_text.strip().startswith("```json"):
            raw_text = raw_text.strip()[7:].strip("`")
        parsed = JobClassificationTable.model_validate_json(raw_text) if raw_text else None
except Exception as e:
    print("❗ Unexpected Responses API error:", e)
    raise

# ---- Save outputs to OUTPUTS_DIR ----
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
raw_path = Path(OUTPUTS_DIR) / "Raw_Response_SINGLE.json"   # was Raw_Response.json

if parsed is not None:
    rows = [row.model_dump() for row in parsed.job_classification_table]
    out_csv = Path(OUTPUTS_DIR) / "Job_Classifications_SINGLE.csv"   # was Job_Classifications.csv
    pd.DataFrame(rows).to_csv(out_csv, index=False, encoding="utf-8")
    out_txt = Path(OUTPUTS_DIR) / "Narrative_SINGLE.txt"             # was Narrative.txt
    out_txt.write_text(parsed.narrative_rationale, encoding="utf-8")
    print("✅ Saved:", out_csv)
    print("✅ Saved:", out_txt)
else:
    print("⚠️ No parsed object returned; saved Raw_Response_SINGLE.json only at:", raw_path)

print("✅ Saved:", raw_path)

# ---- Console visibility ----
print("\n=== RAW JSON STRING FROM MODEL ===")
print(raw_text or "(empty)")
if parsed is not None:
    print("\n=== PARSED (Pydantic) ===")
    print(parsed.model_dump_json(indent=2))

# ---- List run outputs ----
print("\nContents of OUTPUTS_DIR:")
for p in sorted(Path(OUTPUTS_DIR).glob("*")):
    print(" -", p.name)

🤖 Using model: gpt-4o-2024-11-20
✅ Read Ground Truth Masterfile.csv with encoding=cp1252
✅ Saved: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251001_165753/outputs/Job_Classifications_SINGLE.csv
✅ Saved: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251001_165753/outputs/Narrative_SINGLE.txt
✅ Saved: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251001_165753/outputs/Raw_Response_SINGLE.json

=== RAW JSON STRING FROM MODEL ===

{
  "job_classification_table": [
    {
      "job_title_original": "Charter Fiscal Liaison",
      "new_job_title": "Charter Fiscal Analyst II",
      "major_role_group": "Analyst",
      "minor_sub_group": "II",
      "grouping_justification": "The role involves professional accounting duties, financial analysis, and compliance monitoring, which align with the Analyst major role group. The complexity of tasks such as managing charter allocations, conducting risk assessments, and ensuring compliance with governmental accoun

# **Verion 6.0**
>  Pre-flight: are all prerequisites loaded for batch


In [ ]:
# ===== Cell 16.40 — Force input selection (safe auto-finder + explicit override) =====
import os, re, io
import pandas as pd
from pathlib import Path

# OPTIONAL: hard-code your input here to be 100% deterministic:
# BATCH_INPUT_CSV = "/content/drive/MyDrive/Colab Notebooks/Data Inputs/Sample JDs.csv"

REQUIRED_COLS = [
    "Job Description Name",
    "Position Summary",
    "Education",
    "Work Experience",
    "Essential Functions",
    "Licenses and Certifications",
    "Knowledge, Skills and Abilities",
]

def _has_required_cols(df: pd.DataFrame) -> bool:
    cols = {c.strip().lower(): c for c in df.columns}
    need = [c.lower() for c in REQUIRED_COLS]
    return all(n in cols for n in need)

def _read_csv_robust(path: str) -> pd.DataFrame:
    encs = ["utf-8","utf-8-sig","cp1252","latin1","windows-1252","utf-16","utf-16le","utf-16be"]
    seps = [None, ",", "\t", ";", "|"]
    for enc in encs:
        for sep in seps:
            try:
                df = pd.read_csv(path, encoding=enc, sep=sep, engine="python")
                if df.shape[1] >= 1:
                    return df
            except Exception:
                pass
    raw = Path(path).read_bytes()
    for enc in encs:
        try:
            txt = raw.decode(enc, errors="ignore")
            for sep in seps:
                try:
                    df = pd.read_csv(io.StringIO(txt), sep=sep, engine="python")
                    if df.shape[1] >= 1:
                        return df
                except Exception:
                    pass
        except Exception:
            pass
    raise ValueError(f"[16.40] Could not parse CSV: {path}")

def _is_output_like(p: Path) -> bool:
    s = str(p).lower()
    bad_names = ["run results", "/outputs/", "disagreements", "job_classifications_batch",
                 "classified_job_descriptions", "prompt_audit", "run_quality_report"]
    return any(b in s for b in bad_names)

def _auto_find_input_safe():
    roots = [
        Path("/content/drive/MyDrive/Colab Notebooks/Data Inputs"),
        Path("/content/drive/My Drive/Colab Notebooks/Data Inputs"),
        Path("/content/drive/MyDrive/Colab Notebooks"),
        Path("/content/drive/My Drive/Colab Notebooks"),
        Path("/content"),
        Path.cwd(),
    ]
    want = ["sample jds", "new sample", "sample_jds"]
    cands = []
    for r in roots:
        if not r.exists(): continue
        try:
            for p in r.rglob("*.csv"):
                if _is_output_like(p):  # exclude outputs
                    continue
                name = p.name.lower()
                if any(w in name for w in want):
                    cands.append(p)
        except Exception:
            pass
    # newest first
    cands.sort(key=lambda p: p.stat().st_mtime, reverse=True)

    # only accept candidates that have the required columns
    for p in cands:
        try:
            df = _read_csv_robust(str(p))
            if _has_required_cols(df):
                return str(p)
        except Exception:
            continue
    return None

# Resolve the input path
from pathlib import Path
if "BATCH_INPUT_CSV" in globals() and BATCH_INPUT_CSV and Path(BATCH_INPUT_CSV).exists():
    print(f"[16.40] Using explicit BATCH_INPUT_CSV: {BATCH_INPUT_CSV}")
else:
    guess = _auto_find_input_safe()
    if not guess:
        raise FileNotFoundError("[16.40] No valid input CSV found. Put your 'Sample JDs*.csv' in Data Inputs or set BATCH_INPUT_CSV explicitly.")
    BATCH_INPUT_CSV = guess
    print(f"[16.40] Auto-selected input: {BATCH_INPUT_CSV}")

globals()["BATCH_INPUT_CSV"] = BATCH_INPUT_CSV


In [ ]:
# ===== Cell 16.441 — Authoritative input loader (validated df/in_df with row_id) =====
import pandas as pd, io
from pathlib import Path

REQUIRED_COLS = [
    "Job Description Name",
    "Position Summary",
    "Education",
    "Work Experience",
    "Essential Functions",
    "Licenses and Certifications",
    "Knowledge, Skills and Abilities",
]
ALT_NAMES = {
    "job title": "Job Description Name",
    "job_description_name": "Job Description Name",
    "position summary": "Position Summary",
    "knowledge skills and abilities": "Knowledge, Skills and Abilities",
    "knowledge, skills & abilities": "Knowledge, Skills and Abilities",
    "licenses & certifications": "Licenses and Certifications",
    "licenses/certifications": "Licenses and Certifications",
    "experience": "Work Experience",
}

def _read_csv_robust(path: str) -> pd.DataFrame:
    encs = ["utf-8","utf-8-sig","cp1252","latin1","windows-1252","utf-16","utf-16le","utf-16be"]
    seps = [None, ",", "\t", ";", "|"]
    for enc in encs:
        for sep in seps:
            try:
                df = pd.read_csv(path, encoding=enc, sep=sep, engine="python")
                if df.shape[1] >= 1:
                    return df
            except Exception:
                pass
    raw = Path(path).read_bytes()
    for enc in encs:
        try:
            txt = raw.decode(enc, errors="ignore")
            for sep in seps:
                try:
                    df = pd.read_csv(io.StringIO(txt), sep=sep, engine="python")
                    if df.shape[1] >= 1:
                        return df
                except Exception:
                    pass
        except Exception:
            pass
    raise ValueError(f"[16.441] Could not parse CSV: {path}")

if "BATCH_INPUT_CSV" not in globals():
    raise RuntimeError("[16.441] BATCH_INPUT_CSV not set. Run 16.40 first.")
p = Path(BATCH_INPUT_CSV)
if not p.exists():
    raise FileNotFoundError(f"[16.441] Input not found: {p}")

df_in = _read_csv_robust(str(p))

# header canonicalization
ren = {}
for c in df_in.columns:
    key = c.strip().lower()
    if key in ALT_NAMES:
        ren[c] = ALT_NAMES[key]
df_in = df_in.rename(columns=ren)

missing = [c for c in REQUIRED_COLS if c not in df_in.columns]
if missing:
    raise ValueError(f"[16.441] Missing required columns: {missing}. Found={list(df_in.columns)}\nFile: {p}")

# drop fully-empty rows across determinant columns
det_cols = [c for c in REQUIRED_COLS if c != "Job Description Name"]
mask_all_blank = df_in[det_cols].applymap(lambda x: str(x).strip() if pd.notna(x) else "").eq("").all(axis=1)
if mask_all_blank.any():
    before = len(df_in)
    df_in = df_in[~mask_all_blank].copy()
    print(f"[16.441] Dropped {before - len(df_in)} fully-empty rows (determinant columns).")

# ensure row_id
if "row_id" not in df_in.columns:
    df_in = df_in.reset_index().rename(columns={"index":"row_id"})
else:
    try:
        df_in["row_id"] = df_in["row_id"].astype(int)
    except Exception:
        df_in = df_in.reset_index().rename(columns={"index":"row_id"})

print(f"[16.441] Loaded rows: {len(df_in)} from {p}")

# export authoritative inputs
in_df = df_in.copy()
df    = df_in.copy()
globals().update({"in_df": in_df, "df": df})


In [ ]:
# ===== Cell 16.44 — Prompt debug sampler (assumes 16.441 prepared in_df) =====
import pandas as pd

if 'full_text_for_row' not in globals():
    raise RuntimeError("full_text_for_row is not defined. Run Cell 15.351 first.")
if "in_df" not in globals() or not isinstance(in_df, pd.DataFrame) or len(in_df) == 0:
    raise RuntimeError("[16.44] in_df missing/empty. Run 16.40 then 16.441.")

SAMPLE_ROWS = [0, 1, 2]

for ridx in SAMPLE_ROWS:
    if ridx >= len(in_df): continue
    row = in_df.iloc[ridx].copy()
    if 'row_id' not in row.index or pd.isna(row['row_id']):
        row['row_id'] = ridx

    print("="*80)
    print(f"ROW {int(row['row_id'])} — {row.get('Job Description Name','(no name)')}")
    txt = full_text_for_row(row)

    title = str(row.get("Job Description Name","") or "")
    if title and title.lower() in txt.lower():
        print("WARNING: Title is present in prompt — injector may not be active.")

    tail = "\n".join(txt.splitlines()[-40:])
    print(tail)


In [ ]:
# ===== Cell 16.442 — Batch window (make sure we actually process rows) =====
# If you used ROW_LIMIT/ROW_START earlier, this makes sure they won't zero your run.
ROW_START = globals().get("ROW_START", 0)
ROW_LIMIT = globals().get("ROW_LIMIT", None)  # None means "all"
print(f"[16.431] ROW_START={ROW_START} | ROW_LIMIT={ROW_LIMIT} | df_len={len(df) if 'df' in globals() else 'NA'}")


In [ ]:
# ===== Cell 16.45 — Pre-flight: are all prerequisites loaded for batch? =====
from pathlib import Path

print("Have MODEL_ID:", 'MODEL_ID' in globals(), (MODEL_ID if 'MODEL_ID' in globals() else None))
print("Have df:", 'df' in globals(), (len(df) if 'df' in globals() else None))
print("Have zero_shot_prompt:", 'zero_shot_prompt' in globals())
print("Have OUTPUTS_DIR:", 'OUTPUTS_DIR' in globals(), (OUTPUTS_DIR if 'OUTPUTS_DIR' in globals() else None))

if 'RUN_DIR' in globals():
    print("RUN_DIR:", RUN_DIR)
    print("Outputs path will be:", Path(OUTPUTS_DIR) / "Job_Classifications_Batch.csv")
else:
    print("RUN_DIR missing — re-run your unique run cell (Cell 4).")


In [ ]:
from pathlib import Path
(Path(OUTPUTS_DIR)/"Job_Classifications_Batch.csv").unlink(missing_ok=True)
(Path(OUTPUTS_DIR)/"Batch_Errors.json").unlink(missing_ok=True)


# **Version 6.0**
>  Runs batch file  
> Batch v3.1 (DEBUG: loud logs, resume-safe, JSON fence fix)  
> Live peek into processing


In [ ]:
# ===== Cell 16.5 — Batch v3.1 (DEBUG: loud logs, resume-safe, JSON fence fix) =====
from openai import OpenAI
from pathlib import Path
import pandas as pd, json, time, random, inspect, re, shutil

print("=== Batch v3.1 start ===")

# ---- prerequisites ----
assert 'df' in globals(), "Run Cell 4 first (loads df)."
assert 'OUTPUTS_DIR' in globals(), "Run the unique-run cell first."
assert 'MODEL_ID' in globals(), "Run Environment Setup first."
assert 'zero_shot_prompt' in globals(), "Define zero_shot_prompt (your prompt cell)."

print("MODEL_ID:", MODEL_ID)
print("Rows in df:", len(df))
print("OUTPUTS_DIR:", OUTPUTS_DIR)

# If available, show SDK version
try:
    import openai as _o
    print("openai SDK:", getattr(_o, "__version__", "(unknown)"))
except Exception:
    pass

client = OpenAI(timeout=60.0, max_retries=2)

# ---- robust CSV reader if you have it from Cell 4 ----
def _read_csv(path: Path, **kw):
    if 'read_csv_smart' in globals():
        return read_csv_smart(path, **kw)
    return pd.read_csv(path, encoding="utf-8", **kw)

# ---- tiny context from Ground Truth (once) ----
context_block = ""
gt_path = Path(INPUTS_DIR) / "Ground Truth Masterfile.csv" if 'INPUTS_DIR' in globals() else None
if gt_path and gt_path.exists():
    try:
        gt_df = _read_csv(gt_path).fillna("")
        preferred_cols = [
            "Original Job Title","New Job Title","Major Role Group","Minor Sub-Group","Justification for Grouping",
            "Position Summary","Education","Work Experience","Licenses and Certifications","Essential Functions","Knowledge, Skills and Abilities"
        ]
        cols = [c for c in preferred_cols if c in gt_df.columns] or list(gt_df.columns)[:8]
        mini = gt_df[cols].head(3)
        context_block = "Context (3 ground-truth examples):\n" + mini.to_json(orient="records", force_ascii=False)
        print("Context block chars:", len(context_block))
    except Exception as e:
        context_block = f"(Context unavailable: {e})"
        print("Context build error:", e)
else:
    print("No Ground Truth CSV found at", gt_path)

def build_job_text(r):
    def getv(col):
        try:
            v = r[col]
            return "" if pd.isna(v) else str(v)
        except Exception:
            return ""
    return f"""Job Description Name: {getv('Job Description Name')}

Position Summary: {getv('Position Summary')}
Education: {getv('Education')}
Work Experience: {getv('Work Experience')}
Licenses and Certifications: {getv('Licenses and Certifications')}
Essential Functions: {getv('Essential Functions')}
Knowledge, Skills and Abilities: {getv('Knowledge, Skills and Abilities')}
"""

def full_text_for_row(r):
    return (
        zero_shot_prompt.strip()
        + ("\n\n" + context_block if context_block else "")
        + "\n\nClassify the following job description:\n\n"
        + build_job_text(r)
    )

# ---- capability detection ----
def _has_param(obj, name: str) -> bool:
    try:
        return name in inspect.signature(obj).parameters
    except Exception:
        return False

supports_parse_schema  = _has_param(client.responses.parse,  "response_format")
supports_create_schema = _has_param(client.responses.create, "response_format")

print("supports_parse_schema:", supports_parse_schema,
      "| supports_create_schema:", supports_create_schema)

# ---- JSON sanitizers (strip ```json fences etc.) ----
_fence_re = re.compile(r"^\s*```(?:json)?\s*(.*?)\s*```\s*$", re.DOTALL|re.IGNORECASE)
_brace_re = re.compile(r"\{.*\}", re.DOTALL)

def coerce_to_json_str(raw: str) -> str:
    if not isinstance(raw, str):
        return ""
    s = raw.strip()
    m = _fence_re.match(s)
    if m:
        s = m.group(1).strip()
    if not s.startswith("{"):
        m2 = _brace_re.search(s)
        if m2:
            s = m2.group(0)
    return s

# ---- call wrapper ----
def call_model_with_text(text, temp, max_tokens):
    if supports_parse_schema:
        resp = client.responses.parse(
            model=MODEL_ID,
            input=[{"role": "user", "content": [{"type":"input_text","text": text}]}],
            temperature=temp,
            max_output_tokens=max_tokens,
            response_format=JobClassificationTable,
        )
        return resp.output_parsed, resp.output_text or ""
    elif supports_create_schema:
        schema = JobClassificationTable.model_json_schema()
        resp = client.responses.create(
            model=MODEL_ID,
            input=[{"role": "user", "content": [{"type":"input_text","text": text}]}],
            temperature=temp,
            max_output_tokens=max_tokens,
            response_format={
                "type":"json_schema",
                "json_schema":{"name":"JobClassificationTable","schema":schema,"strict":True},
            },
        )
        raw = getattr(resp, "output_text", None) or ""
        try:
            return JobClassificationTable.model_validate_json(raw), raw
        except Exception:
            cleaned = coerce_to_json_str(raw)
            return JobClassificationTable.model_validate_json(cleaned), cleaned
    else:
        schema_json = json.dumps(JobClassificationTable.model_json_schema(), indent=2)
        strict = (
            "You MUST return ONLY valid JSON that matches the following JSON Schema. No prose, no markdown.\n"
            f"JSON Schema:\n{schema_json}\n\nTask:\n{text}"
        )
        resp = client.responses.create(
            model=MODEL_ID,
            input=[{"role":"user","content":[{"type":"input_text","text": strict}]}],
            temperature=temp,
            max_output_tokens=max_tokens,
        )
        raw = getattr(resp, "output_text", None) or ""
        cleaned = coerce_to_json_str(raw)
        return JobClassificationTable.model_validate_json(cleaned), cleaned

def backoff_sleep(k): time.sleep(min(20, 1.8**k + random.random()))

# ---- batching parameters (start with a small limit to confirm) ----
ROW_START   = 0
ROW_LIMIT   = None          # ← first test; set to None after you see progress
TEMP        = 0.2
MAX_TOKENS  = 900
SAVE_EVERY  = 2
MAX_ATTEMPTS_PER_ROW = 3

# ---- resume: skip rows already saved ----
batch_csv_path = Path(OUTPUTS_DIR) / "Job_Classifications_Batch.csv"
processed = set()
if batch_csv_path.exists():
    try:
        prior = pd.read_csv(batch_csv_path, usecols=["source_row_index"])
        processed = set(prior["source_row_index"].astype(int).tolist())
        print(f"Resume mode: {len(processed)} rows already done; will skip them.")
    except Exception as e:
        print("Resume disabled (could not read prior batch CSV):", e)

# ---- plan iteration ----
end_idx = len(df) if ROW_LIMIT is None else min(len(df), ROW_START + ROW_LIMIT)
indexes = [i for i in range(ROW_START, end_idx) if i not in processed]
print(f"Planned rows to process: {len(indexes)} of {len(df)} (from {ROW_START} to {end_idx-1})")
if not indexes:
    print("Nothing to do: either ROW_LIMIT=0, or all planned rows already in batch CSV,")
    print("or ROW_START >= end_idx. If you want a clean re-run, delete previous batch files:")
    print(" (Path(OUTPUTS_DIR)/'Job_Classifications_Batch.csv').unlink(missing_ok=True)")
    print(" (Path(OUTPUTS_DIR)/'Batch_Errors.json').unlink(missing_ok=True)")

records, errors = [], []
start_time = time.time()

# ---- loop ----
for k, i in enumerate(indexes, start=1):
    r = df.iloc[i]
    text = full_text_for_row(r)

    t0 = time.time()
    parsed = None
    raw    = ""

    for attempt in range(MAX_ATTEMPTS_PER_ROW):
        try:
            parsed, raw = call_model_with_text(text, TEMP, MAX_TOKENS)
            break
        except Exception as e:
            msg = str(e)
            if attempt == MAX_ATTEMPTS_PER_ROW - 1:
                snippet = (coerce_to_json_str(raw) if raw else "")[:600]
                errors.append((i, "exception", msg[:500], snippet))
            backoff_sleep(attempt)

    if parsed:
        try:
            for rec in parsed.job_classification_table:
                row_out = rec.model_dump()
                row_out["source_row_index"] = i
                row_out["model_used"] = MODEL_ID
                records.append(row_out)
        except Exception as e:
            errors.append((i, "parse_collect_error", str(e)[:300], (raw or "")[:300]))
    else:
        cleaned = coerce_to_json_str(raw) if raw else ""
        errors.append((i, "no_parsed_output", cleaned[:600]))

    # checkpoint save
    if (k % SAVE_EVERY == 0) or (k == len(indexes)):
        if records:
            if batch_csv_path.exists():
                try:
                    prev = pd.read_csv(batch_csv_path)
                    merged = pd.concat([prev, pd.DataFrame(records)], ignore_index=True)
                    merged.drop_duplicates(subset=["source_row_index","job_title_original","new_job_title"], inplace=True)
                    merged.to_csv(batch_csv_path, index=False, encoding="utf-8")
                except Exception:
                    pd.DataFrame(records).to_csv(batch_csv_path, index=False, encoding="utf-8")
            else:
                pd.DataFrame(records).to_csv(batch_csv_path, index=False, encoding="utf-8")
            print(f"Checkpoint: wrote {len(pd.read_csv(batch_csv_path))} rows to batch CSV.")
            records = []
        Path(OUTPUTS_DIR, "Batch_Errors.json").write_text(json.dumps(errors, indent=2), encoding="utf-8")

    print(f"[{k}/{len(indexes)}] row {i} in {time.time()-t0:.1f}s | total {(time.time()-start_time)/60:.1f} min | "
          f"ok so far {k - len(errors)} | err {len(errors)}")

# copy batch → single so housekeeping/master sees it
if batch_csv_path.exists():
    dst = Path(OUTPUTS_DIR) / "Job_Classifications.csv"
    shutil.copy2(batch_csv_path, dst)
    print("📄 Copied batch to:", dst)

print("✅ Batch complete. Files in:", OUTPUTS_DIR)


In [ ]:
# ===== Cell 16.54 — Live peek while batch runs =====
from pathlib import Path
import pandas as pd

p = Path(OUTPUTS_DIR) / "Job_Classifications_Batch.csv"
if p.exists():
    dfb = pd.read_csv(p)
    print("Rows saved so far:", len(dfb))
    # show last few and a quick look at which source rows are pending
    display(dfb.tail(5))
    if "source_row_index" in dfb.columns and 'df' in globals():
        done = set(dfb["source_row_index"].astype(int))
        pending = [i for i in range(len(df)) if i not in done]
        print("Remaining rows:", len(pending), "| next up:", pending[:10])
else:
    print("No batch file yet at:", p)


# **Version 6.0**
> Batch audit: counts, parameters, error preview   
> Sanity Check  

In [ ]:
# ===== Cell 16.55 — Batch audit: counts, parameters, error preview =====
from pathlib import Path
import pandas as pd, json

assert 'OUTPUTS_DIR' in globals(), "Run Cell 4 first (creates OUTPUTS_DIR)."
assert 'df' in globals(), "Run Cell 4 first (loads df)."

print("Total rows in input df:", len(df))

batch_csv_path = Path(OUTPUTS_DIR) / "Job_Classifications_Batch.csv"
if batch_csv_path.exists():
    dfb = pd.read_csv(batch_csv_path)
    print("Rows saved in batch CSV:", len(dfb))
    if "source_row_index" in dfb.columns:
        done = sorted(dfb["source_row_index"].astype(int).unique().tolist())
        print("First 10 processed row indexes:", done[:10])
        print("Last 10 processed row indexes:", done[-10:])
    else:
        print("Note: 'source_row_index' column missing in batch CSV.")
else:
    print("⚠️ No batch CSV found at:", batch_csv_path)

errors_path = Path(OUTPUTS_DIR) / "Batch_Errors.json"
if errors_path.exists():
    try:
        errs = json.loads(errors_path.read_text())
        print("Error entries:", len(errs))
        for j, e in enumerate(errs[:5]):
            print(f"  {j+1}.", e if isinstance(e, str) else (e[0:2] if isinstance(e, list) else e))
    except Exception as e:
        print("Could not read Batch_Errors.json:", e)
else:
    print("No Batch_Errors.json present — either none failed or nothing ran.")


In [ ]:
# ===== Cell 16.565 — Deterministic overrides (NO TITLE heuristics; KSAC-text only) =====
import re, glob
from pathlib import Path
import pandas as pd

OUTPUTS_DIR = Path(globals().get("OUTPUTS_DIR","./outputs"))
ROLE_SYNONYMS = globals().get("ROLE_SYNONYMS", {})
VALID_ROLES   = set(globals().get("VALID_ROLES", []))

def _latest(globs):
    hits=[]
    for g in globs: hits += glob.glob(str(OUTPUTS_DIR / g))
    hits = sorted(hits, key=lambda p: Path(p).stat().st_mtime, reverse=True)
    return Path(hits[0]) if hits else None

pred_path = _latest(["*canon3.csv","*canon2.csv","*canon.csv",
                     "Job_Classifications_Batch*.csv","classified_job_descriptions*.csv"])
if not pred_path: raise FileNotFoundError("No predictions found; run 16.5 and 16.56 first.")
print("[16.565] Reading:", pred_path)

df = pd.read_csv(pred_path, engine="python")
def _norm(s): return re.sub(r"[^a-z0-9]+","", str(s).lower())

def _get_col(df, *cands):
    want = {_norm(c) for c in cands}
    for c in df.columns:
        if _norm(c) in want: return c
    for c in df.columns:
        n = _norm(c)
        if any(t in n for t in want): return c
    return None

major = _get_col(df, "major_role_group","major role group","major")
minor = _get_col(df, "minor_sub_group","minor sub-group","minor subgroup","minor","level")
ksacs = _get_col(df, "knowledge, skills and abilities","ksacs","knowledge skills and abilities")
efunc = _get_col(df, "essential functions")
psumm = _get_col(df, "position summary")

print("[16.565] Columns (no title used):",
      "\n  major:", major, "\n  minor:", minor,
      "\n  ksacs:", ksacs, "\n  essential:", efunc, "\n  summary:", psumm)

if not major: raise ValueError("[16.565] Could not resolve the major role column.")

def _s(x):
    try: return "" if pd.isna(x) else str(x).strip()
    except Exception: return (str(x) if x is not None else "").strip()

def _has(text, pats):
    t = _s(text).lower()
    return any(re.search(p, t) for p in pats)

# Patterns (match inside KSAC/EFUNC/SUMMARY only)
P_TEACHER     = [r"\bteacher\b", r"\binstructor\b"]
P_COACH       = [r"\bcoach\b", r"instructional coach", r"\bplc\b",
                 r"coaching cycles", r"professional development", r"restorative practice", r"restorative practices", r"\btransition\b", r"\bjdc\b"]
P_AP          = [r"\bassistant principal\b", r"\basst principal\b", r"\bassistant\-principal\b", r"\ba\.?p\.?\b(?!\w)"]
P_PRINCIPAL   = [r"\bprincipal\b(?!\s*assistant)"]
P_SUPERVISOR  = [r"\bsupervisor\b", r"\bsupervise\b", r"\bfront[- ]line\b"]
P_MGR_CREW    = [r"\b(technicians?|crew|field|maintenance|repair)\b"]
P_EXEC_DIR    = [r"\bexecutive director\b", r"\bexec\.?\s*director\b", r"\bexecutive dir\b", r"\bexec dir\b"]
P_TRANSLATOR  = [r"\btranslator\b", r"\binterpreter\b", r"language access", r"bilingual", r"multilingual"]

P_BAD_LEVEL   = [r"\bIV\b|\b4\b|\bV\b|\b5\b|\bVI\b|\b6\b"]
P_LEAD_TOK    = [r"\blead\b"]
P_ROMAN_I     = [r"\bI\b(?![A-Z])"]; P_ROMAN_II=[r"\bII\b(?![A-Z])"]; P_ROMAN_III=[r"\bIII\b(?![A-Z])"]
P_ARABIC_1    = [r"\b1\b"]; P_ARABIC_2=[r"\b2\b"]; P_ARABIC_3=[r"\b3\b"]

def _blob(row):
    parts = [psumm, efunc, ksacs]
    return " ".join(_s(row.get(c)) for c in parts if c).lower()

def _level_from_blob(row):
    t = _blob(row)
    if _has(t, P_LEAD_TOK): return "Lead"
    if _has(t, P_ROMAN_III) or _has(t, P_ARABIC_3): return "III"
    if _has(t, P_ROMAN_II)  or _has(t, P_ARABIC_2): return "II"
    if _has(t, P_ROMAN_I)   or _has(t, P_ARABIC_1): return "I"
    return None

def _norm_minor(val, row):
    s = _s(val).upper()
    if s in {"LEAD","I","II","III"}: return "Lead" if s=="LEAD" else s
    if _has(s, P_BAD_LEVEL): return "Lead"
    hint = _level_from_blob(row)
    return hint or "I"

ALLOWED_FORCE = {"Teacher","Coach","Principal","Assistant Principal","Supervisor",
                 "Executive Director","Translator","Director","Specialist","Manager"}

def _force_role(target, cur_major):
    if target in VALID_ROLES: return target
    low = target.lower()
    for k,v in (ROLE_SYNONYMS or {}).items():
        if k in low and v in VALID_ROLES: return v
    return target if target in ALLOWED_FORCE else cur_major

fix_ct_major = fix_ct_minor = 0
notes = []

for i, row in df.iterrows():
    cur_major = _s(row.get(major))
    t_blob    = _blob(row)

    target = None
    if _has(t_blob, P_AP):                                   target = "Assistant Principal"
    elif _has(t_blob, P_PRINCIPAL):                          target = "Principal"
    elif _has(t_blob, P_EXEC_DIR):                           target = "Executive Director"
    elif _has(t_blob, P_TRANSLATOR):                         target = "Translator"
    elif _has(t_blob, P_COACH):                              target = "Coach"
    elif _has(t_blob, P_SUPERVISOR) and _has(t_blob, P_MGR_CREW): target = "Supervisor"
    elif _has(t_blob, P_TEACHER):                            target = "Teacher"

    if target and target != cur_major:
        chosen = _force_role(target, cur_major)
        if chosen != cur_major:
            df.at[i, major] = chosen
            fix_ct_major += 1
            if len(notes) < 18:
                notes.append(f"row {i}: {cur_major} → {chosen} (KSAC-text override)")

    if minor:
        new_m = _norm_minor(row.get(minor), row)
        if new_m != _s(row.get(minor)):
            df.at[i, minor] = new_m
            fix_ct_minor += 1

out_path = pred_path.with_name(pred_path.stem + "_canonNoTitle.csv")
df.to_csv(out_path, index=False, encoding="utf-8")
print(f"[16.565] Major fixes: {fix_ct_major} | Level fixes: {fix_ct_minor}")
if notes:
    print("[16.565] Sample fixes:")
    for msg in notes: print("  -", msg)
print("[16.565] Wrote ->", out_path, "(no title heuristics)")


In [ ]:
# ===== Cell 16.5655 — Strong title-phrase scrub in justifications =====
import re, glob
from pathlib import Path
import pandas as pd

OUTPUTS_DIR = Path(globals().get("OUTPUTS_DIR","./outputs"))

def _latest(globs):
    hits=[]; [hits.extend(glob.glob(str(OUTPUTS_DIR / g))) for g in globs]
    hits = sorted(hits, key=lambda p: Path(p).stat().st_mtime, reverse=True)
    return Path(hits[0]) if hits else None

pred_path = _latest(["*scrubbed.csv","*canonNoTitle.csv","*canon*.csv",
                     "Job_Classifications_Batch*.csv","classified_job_descriptions*.csv"])
if not pred_path: raise FileNotFoundError("No predictions found to scrub.")
df = pd.read_csv(pred_path, engine="python")

# locate columns
def _norm(s): return re.sub(r"[^a-z0-9]+","", str(s).lower())
just_col = next((c for c in df.columns if _norm(c) in {"groupingjustification","justification","rationale"}), None)
title_col = next((c for c in df.columns if _norm(c) in {"jobdescriptionname","originaljobtitle"}), None)

if not just_col:
    print("[16.5655] No justification column; skipping.")
else:
    pat_generic = re.compile(r"\b(based on|from|according to)\s+(the\s+)?(job\s*title|title)\b", re.I)
    df[just_col] = df[just_col].astype(str).str.replace(pat_generic, "[redacted: no-title policy]", regex=True)

    # also scrub any literal appearances of the actual job title text
    if title_col:
        titles = df[title_col].astype(str).fillna("").tolist()
        # dedupe and sort by length to avoid partial overlaps
        titles = sorted(set([t for t in titles if t.strip()]), key=len, reverse=True)
        for t in titles[:200]:  # safety cap
            try:
                esc = re.escape(t)
                df[just_col] = df[just_col].str.replace(rf"\b{esc}\b", "[redacted title]", regex=True)
            except Exception:
                pass

    out = pred_path.with_name(pred_path.stem + "_scrubbed.csv")
    df.to_csv(out, index=False, encoding="utf-8")
    print("[16.5655] Wrote ->", out)


In [ ]:
# ===== Cell 16.6 — Quick sanity check for current run =====
from pathlib import Path
import pandas as pd, json

assert 'OUTPUTS_DIR' in globals(), "Run your unique-run cell first (defines OUTPUTS_DIR)."

batch = Path(OUTPUTS_DIR) / "Job_Classifications_Batch.csv"
if batch.exists():
    dfb = pd.read_csv(batch)
    print("✅ Batch rows in this run:", len(dfb))
    display(dfb.head(5))
else:
    print("⚠️ No batch file found at", batch)

errs = Path(OUTPUTS_DIR) / "Batch_Errors.json"
if errs.exists():
    e = json.loads(Path(errs).read_text())
    print("⚠️ Rows with errors:", len(e))
    if e:
        print("First error:", e[0])


In [ ]:
# ===== Cell 16.965 — justification title-phrase scanner =====
import re, glob
from pathlib import Path
import pandas as pd

OUTPUTS_DIR = Path(globals().get("OUTPUTS_DIR","./outputs"))
def _latest(globs):
    hits=[]; [hits.extend(glob.glob(str(OUTPUTS_DIR / g))) for g in globs]
    hits = sorted(hits, key=lambda p: Path(p).stat().st_mtime, reverse=True)
    return Path(hits[0]) if hits else None

pred_path = _latest(["*scrubbed.csv","*canonNoTitle.csv","*canon4.csv","*canon3.csv","*canon2.csv","*canon.csv",
                     "Job_Classifications_Batch*.csv","classified_job_descriptions*.csv"])
print("[16.965] Scanning:", pred_path)

df = pd.read_csv(pred_path, engine="python")
def _norm(s): return re.sub(r"[^a-z0-9]+","", str(s).lower())
just = next((c for c in df.columns if _norm(c) in {"groupingjustification","justification","rationale"}), None)
if not just:
    print("[16.965] No justification column found.");
else:
    pat = re.compile(r"\b(based on|from|according to)\s+(the\s+)?(job\s*title|title)\b", re.I)
    hits = df[df[just].astype(str).str.contains(pat)]
    print(f"[16.965] Title-justification hits: {len(hits)}")
    if len(hits):
        display(hits[[just]].head(10))


In [ ]:
# ===== Cell 16.97 — Run Summary & Quality Report (robust align + canonical roles) =====
import os, io, json, re, math, pandas as pd, numpy as np
from pathlib import Path
from datetime import datetime

OUTPUTS_DIR = Path(globals().get("OUTPUTS_DIR", "./outputs"))

# -------- helpers --------
def _read_table(path: str) -> pd.DataFrame:
    encs = ["utf-8","utf-8-sig","cp1252","latin1","windows-1252","utf-16","utf-16le","utf-16be"]
    seps = [None, ",", "\t", ";", "|"]
    for enc in encs:
        for sep in seps:
            try:
                df = pd.read_csv(path, encoding=enc, sep=sep, engine="python")
                if df.shape[1] >= 1:
                    return df
            except Exception:
                pass
    # last-chance
    return pd.read_csv(path, engine="python")

def _cols(df): return {c.lower(): c for c in df.columns}

# Canonicalize majors to MNPS role names only (strip level and synonyms).
ROLE_SYNONYMS = {
    "instructor": "Teacher",
    "exec dir": "Executive Director",
    "executive dir": "Executive Director",
    "ap": "Assistant Principal",
}
VALID_ROLES = set(globals().get("VALID_ROLES", [])) or {
    "Director","Manager","Supervisor","Specialist","Analyst","Technician","Advisor",
    "Teacher","Coach","Liaison","Architect","Designer","Principal","Executive Director",
    "Lead Tech","Coordinator","Accountant","Partner","Assistant Principal","Translator"
}

LEVEL_TOKS = {"lead","i","ii","iii"}

def canon_role(s: str) -> str:
    s = (s or "").strip()
    low = s.lower()
    # strip trailing roman numerals / "lead" tokens if the model leaked them into major
    parts = [p for p in re.split(r"[ /-]+", low) if p]
    parts = [p for p in parts if p not in LEVEL_TOKS]
    low2 = " ".join(parts)
    # collapse synonyms
    for k,v in ROLE_SYNONYMS.items():
        if k in low2:
            return v
    # title-case if it’s already a valid role
    for vr in VALID_ROLES:
        if re.fullmatch(rf"{re.escape(vr)}", s, flags=re.I):
            return vr
    # best-effort: capitalize words
    return " ".join(w.capitalize() for w in low2.split())

def _safe_mean(vals):
    vals = [v for v in vals if isinstance(v, (int,float)) and not math.isnan(v)]
    return sum(vals)/len(vals) if vals else float("nan")

# -------- locate predictions --------
cands = [
    OUTPUTS_DIR / "classified_job_descriptions_refined.csv",
    OUTPUTS_DIR / "classified_job_descriptions.csv",
]
# prefer newest canon artifacts
for pat in ["*canonNoTitle_scrubbed.csv","*canonNoTitle.csv","*canon5.csv","*canon4.csv","*canon3.csv","*canon2.csv","*canon.csv",
            "Job_Classifications_Batch*.csv"]:
    cands += sorted(OUTPUTS_DIR.glob(pat), key=lambda p: p.stat().st_mtime, reverse=True)

pred_path = next((str(p) for p in cands if Path(p).exists()), None)
if not pred_path:
    raise FileNotFoundError("No predictions file found in outputs/.")
pred_df = _read_table(pred_path)

# resolve columns
pc = _cols(pred_df)
major_col = pc.get("major_role_group") or pc.get("major role group") or pc.get("major")
minor_col = pc.get("minor_sub_group") or pc.get("minor sub-group") or pc.get("minor") or pc.get("level")
title_col = pc.get("job description name") or pc.get("original_job_title") or pc.get("original job title")
just_col  = pc.get("grouping_justification") or pc.get("justification") or pc.get("rationale")
if not major_col: raise ValueError("Predictions must include major role column.")

# row ids in predictions (create if missing)
if "row_id" not in pred_df.columns:
    pred_df = pred_df.reset_index().rename(columns={"index":"row_id"})

# -------- load confidence shortlist --------
conf_json = OUTPUTS_DIR / "role_confidence_top5.json"
conf_map_raw = {}
if conf_json.exists():
    try:
        conf_map_raw = json.loads(conf_json.read_text(encoding="utf-8"))
    except Exception:
        conf_map_raw = {}

# normalize conf_map -> {int_row_id: [(canon_role, conf), ...]}
conf_map = {}
for k, lst in conf_map_raw.items():
    try:
        rk = int(k)
    except Exception:
        continue
    pairs = []
    for d in (lst or []):
        role = canon_role(d.get("role",""))
        try:
            conf = float(d.get("confidence")) if d.get("confidence") is not None else None
        except Exception:
            conf = None
        if role:
            pairs.append((role, conf))
    conf_map[rk] = pairs

# -------- compute hit@k with robust alignment --------
def _hit_k(row_id, chosen_role, k=1):
    chosen = canon_role(chosen_role)
    lst = conf_map.get(row_id, [])
    top = [r for r,_ in lst[:k]]
    return chosen in top

def _sel_conf(row_id, chosen_role):
    chosen = canon_role(chosen_role)
    for r,c in conf_map.get(row_id, []):
        if r == chosen:
            return c if isinstance(c,(int,float)) else float("nan")
    return float("nan")

work = pred_df.copy()
# try direct alignment first
if conf_map and len(conf_map) == len(work):
    # If keys 0..n-1 exist but don't match work.row_id values, try positional fallback
    pred_ids = list(work["row_id"])
    conf_keys = sorted(conf_map.keys())
    direct_ok = set(pred_ids) == set(conf_keys)
    if not direct_ok:
        # build position-based map
        conf_pos = {int(i): conf_map[k] for i, k in enumerate(conf_keys)}
        def _hit_pos(idx, chosen, k):
            lst = conf_pos.get(int(idx), [])
            top = [r for r,_ in lst[:k]]
            return canon_role(chosen) in top
        def _sel_pos(idx, chosen):
            lst = conf_pos.get(int(idx), [])
            cr = canon_role(chosen)
            for r,c in lst:
                if r == cr:
                    return c if isinstance(c,(int,float)) else float("nan")
            return float("nan")
        work["_hit1"] = [ _hit_pos(i, r, 1) for i,r in enumerate(work[major_col]) ]
        work["_hit3"] = [ _hit_pos(i, r, 3) for i,r in enumerate(work[major_col]) ]
        work["_sel_conf"] = [ _sel_pos(i, r) for i,r in enumerate(work[major_col]) ]
    else:
        work["_hit1"] = work.apply(lambda r: _hit_k(int(r["row_id"]), r[major_col], 1), axis=1)
        work["_hit3"] = work.apply(lambda r: _hit_k(int(r["row_id"]), r[major_col], 3), axis=1)
        work["_sel_conf"] = work.apply(lambda r: _sel_conf(int(r["row_id"]), r[major_col]), axis=1)
else:
    # no conf_map or different lengths: compute nothing
    work["_hit1"] = False
    work["_hit3"] = False
    work["_sel_conf"] = float("nan")

# -------- optional: Expected pass rate (if present) --------
exp_col = pc.get("expected clarification") or pc.get("expected") or pc.get("notes")
def _parse_expected(txt: str):
    if not isinstance(txt, str): txt = str(txt or "")
    ROLES = list(VALID_ROLES)
    LVLS = ["Lead","I","II","III"]
    roles = [r for r in ROLES if re.search(rf"\b{re.escape(r)}\b", txt, flags=re.I)]
    lvls  = [lv for lv in LVLS if re.search(rf"\b{lv}\b", txt, flags=re.I)]
    return roles, lvls

def _pass_row(row):
    if not exp_col: return True
    exp = str(row.get(exp_col,"") or "").strip()
    if exp == "": return True
    roles, lvls = _parse_expected(exp)
    ok_role = True if not roles else any(canon_role(row[major_col]) == canon_role(r) for r in roles)
    ok_lvl  = True if not lvls else (str(row.get(minor_col,"")).strip().upper() in {lv.upper() for lv in lvls})
    return ok_role and ok_lvl

work["_pass"] = work.apply(_pass_row, axis=1)
pass_rate = float(work["_pass"].mean())

# -------- metrics + report --------
n = len(work)
hit1 = float(work["_hit1"].mean()) if len(work) else float("nan")
hit3 = float(work["_hit3"].mean()) if len(work) else float("nan")
avg_sel_conf = float(_safe_mean(work["_sel_conf"]))

lines = []
lines.append(f"# Run Quality Report — {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')} UTC\n")
lines.append(f"- Predictions: `{pred_path}`")
lines.append(f"- Records: {n}")
lines.append(f"- Confidence shortlist: {'present' if bool(conf_map) else 'absent'}")
if bool(conf_map):
    lines.append(f"- hit@1 (chosen role == top1): {hit1:.3f}")
    lines.append(f"- hit@3 (chosen role ∈ top3):  {hit3:.3f}")
    if avg_sel_conf == avg_sel_conf:
        lines.append(f"- Avg confidence of chosen role: {avg_sel_conf:.3f}")
lines.append(f"- Pass rate vs Expected (blank = pass): {pass_rate:.3f}")

# Save artifacts
(report_path := OUTPUTS_DIR / "run_quality_report.md").write_text("\n".join(lines), encoding="utf-8")
work[["row_id", major_col, minor_col, "_hit1","_hit3","_sel_conf","_pass"]].to_csv(OUTPUTS_DIR/"hit_details.csv", index=False, encoding="utf-8")

print("[Report] Wrote:", report_path)
print("[Report] hit@1:", f"{hit1:.3f}" if hit1==hit1 else "NA", "hit@3:", f"{hit3:.3f}" if hit3==hit3 else "NA")


In [ ]:
# ===== Cell 16.979 — Hard gate: fail on title leak OR zero hit@1 with non-empty shortlist =====
import re, io, json, math, pandas as pd, numpy as np
from pathlib import Path
from datetime import datetime

OUTPUTS_DIR = Path(globals().get("OUTPUTS_DIR","./outputs"))
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# ---------- helpers ----------
def _read_table(path: Path) -> pd.DataFrame:
    encs = ["utf-8","utf-8-sig","cp1252","latin1","windows-1252","utf-16","utf-16le","utf-16be"]
    seps = [None, ",", "\t", ";", "|"]
    for enc in encs:
        for sep in seps:
            try:
                df = pd.read_csv(path, encoding=enc, sep=sep, engine="python")
                if df.shape[1] >= 1:
                    return df
            except Exception:
                pass
    # last-chance
    try:
        txt = path.read_text(errors="ignore")
        return pd.read_csv(io.StringIO(txt), engine="python")
    except Exception:
        return pd.DataFrame()

def _cols(df): return {c.lower(): c for c in df.columns}

VALID_ROLES = set(globals().get("VALID_ROLES", [])) or {
    "Director","Manager","Supervisor","Specialist","Analyst","Technician","Advisor",
    "Teacher","Coach","Liaison","Architect","Designer","Principal","Executive Director",
    "Lead Tech","Coordinator","Accountant","Partner","Assistant Principal","Translator"
}
LEVEL_TOKS = {"lead","i","ii","iii"}
ROLE_SYNONYMS = {
    "instructor": "Teacher",
    "exec dir": "Executive Director",
    "executive dir": "Executive Director",
    "ap": "Assistant Principal",
}

def canon_role(s: str) -> str:
    s = (s or "").strip()
    low = s.lower()
    parts = [p for p in re.split(r"[ /\-]+", low) if p]
    parts = [p for p in parts if p not in LEVEL_TOKS]
    low2 = " ".join(parts)
    for k,v in ROLE_SYNONYMS.items():
        if k in low2: return v
    for vr in VALID_ROLES:
        if re.fullmatch(rf"{re.escape(vr)}", s, flags=re.I): return vr
    for vr in VALID_ROLES:
        if re.match(rf"^{re.escape(vr)}\b", s, flags=re.I): return vr
    return " ".join(w.capitalize() for w in low2.split())

def _latest_prediction_path() -> Path:
    patterns = ["*scrubbed.csv","*canonNoTitle.csv","*canon*.csv",
                "Job_Classifications_Batch*.csv","classified_job_descriptions*.csv"]
    hits = []
    for pat in patterns:
        hits += list(OUTPUTS_DIR.glob(pat))
    if not hits: return None
    hits.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return hits[0]

# ---------- 1) Title leak gate ----------
audit_path = OUTPUTS_DIR / "prompt_audit.csv"
title_leaks = []

if audit_path.exists() and audit_path.stat().st_size > 0:
    audit = _read_table(audit_path)
    cc = _cols(audit)
    title_flag_col = cc.get("title_in_prompt") or cc.get("titleinprompt")
    if title_flag_col and title_flag_col in audit:
        bad = audit[audit[title_flag_col].astype(bool)]
        if not bad.empty:
            title_leaks = bad["row_id"].astype(int).tolist()[:10]
else:
    # Rebuild minimal audit if needed (requires full_text_for_row + in_df or BATCH_INPUT_CSV)
    if "full_text_for_row" in globals():
        def _read_csv_robust(path: str) -> pd.DataFrame:
            for enc in ["utf-8","utf-8-sig","cp1252","latin1","windows-1252","utf-16","utf-16le","utf-16be"]:
                try: return pd.read_csv(path, encoding=enc)
                except Exception: pass
            return pd.read_csv(path, engine="python")
        if "in_df" in globals() and isinstance(in_df, pd.DataFrame) and len(in_df):
            _df = in_df.copy()
        elif "BATCH_INPUT_CSV" in globals() and BATCH_INPUT_CSV and Path(BATCH_INPUT_CSV).exists():
            _df = _read_csv_robust(BATCH_INPUT_CSV)
        else:
            _df = None
        if isinstance(_df, pd.DataFrame) and len(_df):
            if "row_id" not in _df.columns:
                _df = _df.reset_index().rename(columns={"index":"row_id"})
            leaks = []
            for _, r in _df.iterrows():
                title = str(r.get("Job Description Name","") or "")
                txt = full_text_for_row(r)
                if title and title.lower() in txt.lower():
                    leaks.append(int(r["row_id"]))
            title_leaks = leaks[:10]

if title_leaks:
    raise AssertionError(
        f"[HARD GATE] Title leaked into prompts for rows: {title_leaks}. "
        "Prompts must exclude titles. Re-run 15.351 and 15.36, then batch."
    )

print("[16.979] NO title leaks detected in prompts.")

# ---------- 2) hit@1 gate when shortlist exists ----------
conf_json = OUTPUTS_DIR / "role_confidence_top5.json"
shortlist_count = 0
hit1 = None

if conf_json.exists() and conf_json.stat().st_size > 0:
    try:
        conf_raw = json.loads(conf_json.read_text(encoding="utf-8"))
    except Exception:
        conf_raw = {}

    # count non-empty shortlist entries
    for k, lst in conf_raw.items():
        if isinstance(lst, list) and len(lst) > 0:
            shortlist_count += 1

    # load predictions
    pred_path = _latest_prediction_path()
    if pred_path is not None and pred_path.exists():
        df = _read_table(pred_path)
        if "row_id" not in df.columns:
            df = df.reset_index().rename(columns={"index":"row_id"})
        pc = _cols(df)
        major_col = pc.get("major_role_group") or pc.get("major role group") or pc.get("major")
        if major_col:
            # build canonical conf_map
            conf_map = {}
            for k, lst in conf_raw.items():
                if not str(k).isdigit(): continue
                rid = int(k)
                pairs = []
                for d in (lst or []):
                    role = canon_role(d.get("role",""))
                    conf = d.get("confidence", None)
                    try: conf = float(conf) if conf is not None else None
                    except Exception: conf = None
                    if role: pairs.append((role, conf))
                conf_map[rid] = pairs

            # direct alignment? else positional fallback
            if set(df["row_id"]) == set(conf_map.keys()):
                hit1_vals = df.apply(
                    lambda r: canon_role(r[major_col]) in [x for x,_ in conf_map.get(int(r["row_id"]), [])[:1]],
                    axis=1
                ).astype(bool)
            elif len(conf_map) == len(df):
                keys = sorted(conf_map.keys())
                bypos = {i: conf_map[k] for i,k in enumerate(keys)}
                hit1_vals = pd.Series(
                    [ canon_role(df[major_col].iloc[i]) in [x for x,_ in bypos.get(i, [])[:1]]
                      for i in range(len(df)) ],
                    index=df.index
                ).astype(bool)
            else:
                hit1_vals = pd.Series([False]*len(df), index=df.index)

            hit1 = float(hit1_vals.mean()) if len(hit1_vals) else float("nan")

if shortlist_count > 0 and (hit1 is not None) and (not math.isnan(hit1)) and hit1 == 0.0:
    raise RuntimeError(
        "[HARD GATE] hit@1 == 0.0 while the confidence shortlist is non-empty — "
        "this indicates misalignment or canonicalization mismatch. "
        "Ensure you ran 15.2 → 15.25 → 15.201 → 15.351 before the batch, and that 16.97 uses the same canonicalization."
    )

print(f"[16.979] Shortlist entries: {shortlist_count} | hit@1: {('NA' if hit1 is None or math.isnan(hit1) else f'{hit1:.3f}')}")
print("[16.979] Hard gate checks passed.")


In [ ]:
#Cell 17
# Inspect parsed output (Responses API)
try:
    parsed  # from Cell 16
    print(parsed.model_dump_json(indent=2))
except NameError:
    print("No 'parsed' object found. Run Cell 16 first.")


In [ ]:
# Cell 17.5 — Build a response_dict from the Responses API parsed object
from pathlib import Path
import json
import pandas as pd

# Make sure Cell 16 ran (it defines `parsed`) and the run folders exist
assert 'parsed' in globals(), "Run Cell 16 first (it sets `parsed`)."
assert 'OUTPUTS_DIR' in globals(), "Run the unique-run cell first (defines OUTPUTS_DIR)."

# Convert the Pydantic objects to plain dicts
response_dict = {
    "job_classification_table": [rec.model_dump() for rec in parsed.job_classification_table],
    "narrative_rationale": parsed.narrative_rationale,
}

# Optional: preview the first rows
display(pd.DataFrame(response_dict["job_classification_table"]).head(10))

# Optional: save a pretty JSON alongside your other outputs
out_json = Path(OUTPUTS_DIR) / "Parsed_Response.json"
out_json.write_text(json.dumps(response_dict, indent=2), encoding="utf-8")
print("Saved:", out_json)

# Also return the dict so it shows below the cell
response_dict


In [ ]:
#Cell 18
# Preview the saved classifications CSV (if present)
from pathlib import Path
import pandas as pd

csv_path = Path(OUTPUTS_DIR) / "Job_Classifications.csv"
if csv_path.exists():
    display(pd.read_csv(csv_path).head(10))
else:
    print("No Job_Classifications.csv found in", OUTPUTS_DIR)


In [ ]:
# Cell 18.2 — Copy to Google Drive (includes Role Confidence tables)

from google.colab import drive
import os, shutil, datetime
from pathlib import Path

# Mount Drive
drive.mount('/content/drive')

# Base target in Drive
base_target_folder = '/content/drive/My Drive/Colab Notebooks/Run Results'

# Unique run folder
timestamp = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
unique_folder_name = f'RUN_{timestamp}'
target_folder = os.path.join(base_target_folder, unique_folder_name)
os.makedirs(target_folder, exist_ok=True)

# Collect inputs/resources (best effort)
possible_inputs = [
    globals().get("BATCH_INPUT_CSV", "/content/Sample JDs.csv"),
    "/content/MNPS Roles.csv",
    "/content/MNPS KSACs.csv",
    "/content/Competency Extended Descriptions.csv",
    "/content/Korn_Ferry Lominger 38 Competencies.csv",
    globals().get("GROUND_TRUTH_CSV", "/content/Ground Truth Masterfile.csv"),
]

# Collect outputs
OUTPUTS_DIR = globals().get("OUTPUTS_DIR", "./outputs")
out_candidates = [
    os.path.join(OUTPUTS_DIR, "classified_job_descriptions.csv"),
    os.path.join(OUTPUTS_DIR, "classified_job_descriptions_refined.csv"),
    os.path.join(OUTPUTS_DIR, "classification_decision_log.csv"),
    os.path.join(OUTPUTS_DIR, "refinement_log.csv"),
    os.path.join(OUTPUTS_DIR, "role_confidence_indicators.csv"),
    os.path.join(OUTPUTS_DIR, "role_confidence_top5.csv"),
    os.path.join(OUTPUTS_DIR, "role_confidence_top5.json"),
]

files_to_copy = []
for p in possible_inputs + out_candidates:
    if p and os.path.exists(p):
        files_to_copy.append(p)

# Copy
for src in files_to_copy:
    try:
        fname = os.path.basename(src)
        dst = os.path.join(target_folder, fname)
        shutil.copy(src, dst)
        print(f"Copied: {fname}")
    except FileNotFoundError:
        print(f"Missing: {src}")
    except Exception as e:
        print(f"Error copying {src}: {e}")

print("\n[Drive] Copied files to:", target_folder)


In [ ]:
# Cell 19 ===== Housekeeping & Archive (Run Results) =====
# Place this cell at the END of the notebook. Run after your pipeline finishes.
from google.colab import drive
from pathlib import Path
import shutil, json, re
import datetime as dt
import pandas as pd

# ---------- CONFIG (edit to taste) ----------
RUN_ROOT = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
ARCHIVE_DIR = RUN_ROOT / "_archives"
MASTER_DIR  = RUN_ROOT / "_master"

KEEP_LAST_N_RUNS   = 10     # keep this many newest runs; older ones can be deleted
ZIP_OLDER_RUNS     = True   # zip runs (into _archives) to save space
PURGE_RAW_JSON     = True   # delete outputs/Raw_Response.json inside each run
PURGE_PARSED_JSON  = False  # delete outputs/Parsed_Response.json
PURGE_BATCH_ERRORS = False  # delete outputs/Batch_Errors.json
SKIP_CURRENT_RUN   = True   # don't zip/purge/delete the most recent run
DRY_RUN            = True   # <<< safety: set False to actually apply changes

# ---------- Mount Drive (no-op if already mounted) ----------
drive.mount('/content/drive')

# ---------- Helpers ----------
def parse_run_ts(name: str):
    m = re.match(r"RUN_(\d{8}_\d{6})$", name)
    if not m:
        return None
    try:
        return dt.datetime.strptime(m.group(1), "%Y%m%d_%H%M%S")
    except Exception:
        return None

def folder_size_bytes(p: Path) -> int:
    total = 0
    for f in p.rglob("*"):
        if f.is_file():
            try:
                total += f.stat().st_size
            except Exception:
                pass
    return total

def human_mb(nbytes: int) -> str:
    return f"{nbytes/1_000_000:.2f} MB"

# ---------- Discover run folders ----------
runs = []
for d in RUN_ROOT.iterdir():
    if d.is_dir() and d.name.startswith("RUN_"):
        ts = parse_run_ts(d.name)
        if ts:
            runs.append((d, ts))

runs.sort(key=lambda x: x[1], reverse=True)  # newest first
print(f"Found {len(runs)} run folders under: {RUN_ROOT}")

current = runs[0][0] if runs else None
if current:
    print("Most recent run:", current.name)

# Summary of the first few
for d, ts in runs[:5]:
    print(f" - {d.name} | {ts:%Y-%m-%d %H:%M:%S} | size≈ {human_mb(folder_size_bytes(d))}")

# Ensure archive/master dirs
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
MASTER_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Plan actions ----------
actions = []

# 1) Purge large intermediates within runs
def plan_purges(d: Path):
    out = d / "outputs"
    if not out.exists():
        return
    if PURGE_RAW_JSON and (out / "Raw_Response.json").exists():
        actions.append(("delete_file", out / "Raw_Response.json"))
    if PURGE_PARSED_JSON and (out / "Parsed_Response.json").exists():
        actions.append(("delete_file", out / "Parsed_Response.json"))
    if PURGE_BATCH_ERRORS and (out / "Batch_Errors.json").exists():
        actions.append(("delete_file", out / "Batch_Errors.json"))

# 2) Zip older runs (into _archives)
def plan_zip(d: Path):
    z = ARCHIVE_DIR / f"{d.name}.zip"
    if not z.exists():
        actions.append(("zip_folder", (d, z)))

# 3) Delete runs beyond retention
to_prune = runs[KEEP_LAST_N_RUNS:] if KEEP_LAST_N_RUNS is not None else []
for d, ts in runs:
    if SKIP_CURRENT_RUN and current and d == current:
        continue
    # Purges
    plan_purges(d)
    # Zip plan
    if ZIP_OLDER_RUNS:
        plan_zip(d)

for d, ts in to_prune:
    actions.append(("delete_folder", d))

# ---------- Show plan ----------
print("\nPlanned actions:")
if not actions:
    print(" (none)")
else:
    for act, obj in actions:
        if act == "zip_folder":
            d, z = obj
            print(f" - ZIP {d.name}  →  {z.name}")
        else:
            print(f" - {act.upper()}: {obj}")

# ---------- Execute (unless DRY_RUN) ----------
if DRY_RUN:
    print("\nDRY_RUN=True — no changes applied. Set DRY_RUN=False to execute.")
else:
    for act, obj in actions:
        try:
            if act == "delete_file":
                Path(obj).unlink(missing_ok=True)
            elif act == "zip_folder":
                d, z = obj
                # create zip in ARCHIVE_DIR; shutil.make_archive adds .zip automatically
                base_name = z.with_suffix("")  # remove .zip for make_archive
                shutil.make_archive(str(base_name), 'zip', root_dir=d)
            elif act == "delete_folder":
                shutil.rmtree(obj, ignore_errors=True)
        except Exception as e:
            print("  ! Error:", act, obj, e)
    print("\n✅ Housekeeping complete.")

# ---------- Aggregate a master CSV across all runs (safe to do anytime) ----------
frames = []
for d, ts in runs:
    for name in ["Job_Classifications_Batch.csv", "Job_Classifications.csv"]:
        csvp = d / "outputs" / name
        meta = d / "RUN_METADATA.json"
        if csvp.exists():
            try:
                df_run = pd.read_csv(csvp)
                df_run["run_folder"]  = d.name
                df_run["source_file"] = name
                # enrich with metadata if available
                if meta.exists():
                    try:
                        m = json.loads(meta.read_text())
                        df_run["created_utc"] = m.get("created_utc")
                        df_run["model_used"]  = m.get("resolved_model_id") or m.get("model_used")
                    except Exception:
                        pass
                frames.append(df_run)
            except Exception as e:
                print(f"  ! Skipping {csvp.name} due to read error:", e)

if frames:
    master = pd.concat(frames, ignore_index=True)
    MASTER_DIR.mkdir(parents=True, exist_ok=True)
    master_out = MASTER_DIR / "All_Job_Classifications.csv"
    master.to_csv(master_out, index=False, encoding="utf-8")
    print(f"\n📚 Master CSV updated: {master_out} ({len(master)} rows; from {len(frames)} files)")
else:
    print("\n(No job classification CSVs found to aggregate.)")
